# level 5 L1000 annotation

In [1]:
import re
import time
import json
import os
import pandas as pd
from typing import Optional, Dict
import pubchempy as pcp
from rdkit import Chem

import sys
import logging

# Init logger
FMT = '%(asctime)s | [%(levelname)s] %(message)s'
DATEFMT = '%Y-%m-%d %H:%M:%S'
formatter = logging.Formatter(fmt=FMT, datefmt=DATEFMT)

h1 = logging.StreamHandler(sys.stdout)
h1.setLevel(logging.INFO)
h1.addFilter(lambda log: log.levelno == logging.INFO)
h1.setFormatter(formatter)

h2 = logging.StreamHandler(sys.stderr)
h2.setLevel(logging.WARNING)
h2.setFormatter(formatter)

logger = logging.getLogger(__name__)
logger.propagate = False
logger.setLevel(logging.DEBUG)
logger.handlers = [h1, h2]



def is_valid_pubchem_cid(cid) -> bool:
    """Check if pubchem_cid is valid (positive integer)."""
    if cid is None:
        return False
    try:
        return int(cid) > 0
    except (ValueError, TypeError):
        return False


def is_valid_inchikey(inchikey: Optional[str]) -> bool:
    """Check if InChIKey follows standard format (XXXXXXXXXXXXXX-XXXXXXXXXX-X)."""
    if not inchikey or not isinstance(inchikey, str):
        return False
    pattern = r'^[A-Z]{14}-[A-Z]{10}-[A-Z]$'
    return bool(re.match(pattern, inchikey.strip()))


def is_valid_smiles(smiles: str) -> bool:
    """Basic SMILES validation (non-empty string with common SMILES characters)."""
    return Chem.MolFromSmiles(smiles, sanitize=True) is not None


def load_cache_from_json(cache_path: str) -> Dict[str, Optional[int]]:
    """
    Load cache dictionary from JSON file if it exists.
    
    Parameters:
    -----------
    cache_path : str
        Path to the JSON file containing the cache
        
    Returns:
    --------
    Dict[str, Optional[int]]
        Cache dictionary loaded from file, or empty dict if file doesn't exist
    """
    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'r') as f:
                cache = json.load(f)
            # Convert loaded JSON values to Optional[int] type
            # JSON stores None as null, which becomes None in Python
            # Integer values are stored as numbers in JSON, which become Python int when loaded
            cache_typed = {}
            for key, value in cache.items():
                cache_typed[key] = int(value) if value is not None else None
            logger.info(f"Loaded cache from {cache_path} with {len(cache_typed)} entries")
            return cache_typed
        except Exception as e:
            logger.warning(f"Failed to load cache from {cache_path}: {e}. Starting with empty cache.")
            return {}
    else:
        logger.info(f"Cache file {cache_path} not found. Starting with empty cache.")
        return {}


def save_cache_to_json(cache: Dict[str, Optional[int]], cache_path: str) -> None:
    """
    Save cache dictionary to JSON file.
    
    Parameters:
    -----------
    cache : Dict[str, Optional[int]]
        Cache dictionary to save
    cache_path : str
        Path to save the JSON file
    """
    try:
        # Create directory if it doesn't exist
        cache_dir = os.path.dirname(cache_path)
        if cache_dir:  # Only create directory if path contains a directory
            os.makedirs(cache_dir, exist_ok=True)
        
        # Save cache directly to JSON (cache is already JSON-serializable: Dict[str, Optional[int]])
        # None values are preserved as null, integers are stored as numbers (JSON natively supports integers)
        with open(cache_path, 'w') as f:
            json.dump(cache, f, indent=2)
        logger.debug(f"Saved cache to {cache_path} with {len(cache)} entries")
    except Exception as e:
        logger.warning(f"Failed to save cache to {cache_path}: {e}")


def pubchem_mapping_l1000():
    """
    Manual mapping for L1000 compounds where automatic PubChem lookup may fail.
    
    This provides fallback mappings for:
    - Ambiguous compound names
    - Non-standard nomenclature
    - Known problematic lookups
    
    Returns
    -------
    dict
        Dictionary with 'l1000' key containing compound name to PubChem CID mappings
    """
    sm2pubchem = {
        'l1000': {
            # Add manual mappings here as needed
            # Example:
            # 'BRD-K12345678': 123456,
        }
    }
    return sm2pubchem


def _fetch_pubchem_cid_with_retry(identifier: str,
                                  lookup_type: str,
                                  cache: Dict[str, Optional[int]],
                                  cache_key: str,
                                  identifier_label: str,
                                  universal_cache_key: Optional[str] = None,
                                  n_retries: int = 5) -> Optional[int]:
    """
    Helper function to fetch PubChem CID with retry logic.
    
    Parameters:
    -----------
    identifier : str
        The identifier to look up (InChIKey, SMILES, or drug name)
    lookup_type : str
        PubChem lookup type: 'inchikey', 'smiles', or 'name'
    cache : Dict[str, Optional[int]]
        Cache dictionary for storing results
    cache_key : str
        Method-specific key to use in cache dictionary
    identifier_label : str
        Label for logging (e.g., 'InChIKey', 'SMILES', 'drug name')
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to store result under
    n_retries : int
        Number of retries on failure
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    cnt = 0
    cid = None
    while cnt < n_retries:
        try:
            compounds = pcp.get_compounds(identifier, lookup_type)
            cid = compounds[0].cid if compounds else None
            logger.debug("CID for %s '%s': %s", identifier_label, identifier, cid)
            break
        except Exception as e:
            if (isinstance(e, pcp.PubChemHTTPError) or isinstance(e, pcp.TimeoutError) or
                isinstance(e, pcp.ServerError) or isinstance(e, pcp.ServerBusyError)):
                logger.warning("PubChem lookup failed for %s '%s': %s. Retry %d.", 
                             identifier_label, identifier, str(e), cnt)
                cnt += 1
                time.sleep(5)
            else:
                logger.warning("PubChem lookup failed for %s '%s': %s", 
                             identifier_label, identifier, str(e))
                break
    
    # Store in both method-specific and universal cache keys (only if cid is not None)
    if cid is not None:
        cache[cache_key] = cid
        if universal_cache_key:
            cache[universal_cache_key] = cid
    
    return cid


def get_pubchem_cid_by_inchikey(inchikey: str, 
                                 cache: Dict[str, Optional[int]], 
                                 universal_cache_key: Optional[str] = None,
                                 n_retries: int = 5) -> Optional[int]:
    """
    Fetch PubChem CID for a given InChIKey, using cache to skip repeat lookups.
    
    Parameters:
    -----------
    inchikey : str
        InChIKey for mapping to PubChem CID
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to check/store result
    n_retries : int
        The number of retries when connecting to PubChem goes wrong
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    if not is_valid_inchikey(inchikey):
        return None
    
    # Check universal cache first
    if universal_cache_key and universal_cache_key in cache:
        return cache[universal_cache_key]
    
    # Check method-specific cache
    cache_key = f"inchikey:{inchikey}"
    if cache_key in cache:
        return cache[cache_key]
    
    return _fetch_pubchem_cid_with_retry(inchikey, 'inchikey', cache, cache_key, 'InChIKey', 
                                        universal_cache_key, n_retries)


def get_pubchem_cid_by_smiles(smiles: str, 
                              cache: Dict[str, Optional[int]], 
                              universal_cache_key: Optional[str] = None,
                              n_retries: int = 5) -> Optional[int]:
    """
    Fetch PubChem CID for a given SMILES string, using cache to skip repeat lookups.
    
    Parameters:
    -----------
    smiles : str
        SMILES string for mapping to PubChem CID
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to check/store result
    n_retries : int
        The number of retries when connecting to PubChem goes wrong
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    if not is_valid_smiles(smiles):
        return None
    
    # Check universal cache first
    if universal_cache_key and universal_cache_key in cache:
        return cache[universal_cache_key]
    
    # Check method-specific cache
    cache_key = f"smiles:{smiles}"
    if cache_key in cache:
        return cache[cache_key]
    
    return _fetch_pubchem_cid_with_retry(smiles, 'smiles', cache, cache_key, 'SMILES', 
                                        universal_cache_key, n_retries)


def get_pubchem_cid_by_name(drug_name: str, 
                            cache: Dict[str, Optional[int]], 
                            universal_cache_key: Optional[str] = None,
                            n_retries: int = 5) -> Optional[int]:
    """
    Fetch PubChem CID for a given drug name, using cache to skip repeat lookups.
    
    Parameters:
    -----------
    drug_name : str
        Drug name for mapping to PubChem CID
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to check/store result.
        If None, uses drug_name as universal key.
    n_retries : int
        The number of retries when connecting to PubChem goes wrong
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    if pd.isna(drug_name) or not drug_name:
        return None
    
    # Use drug_name as universal key if not provided
    if universal_cache_key is None:
        universal_cache_key = drug_name
    
    # Check universal cache first
    if universal_cache_key in cache:
        return cache[universal_cache_key]
    
    # Check method-specific cache
    cache_key = f"name:{drug_name}"
    if cache_key in cache:
        return cache[cache_key]
    
    return _fetch_pubchem_cid_with_retry(drug_name, 'name', cache, cache_key, 'drug name', 
                                        universal_cache_key, n_retries)


def add_pubchem_cids(df: pd.DataFrame,
                           cache: Dict[str, Optional[int]],
                           pert_id_col: Optional[str] = 'pert_id',
                           drug_col: str = 'perturbagen',
                           pubchem_cid_col: str = 'pubchem_cid',
                           inchikey_col: str = 'inchi_key',
                           smiles_col: str = 'canonical_smiles',
                           cache_path: Optional[str] = None) -> pd.DataFrame:
    """
    Add or update 'pubchem_cid' column to dataframe for L1000 dataset.
    
    Uses multiple strategies in order of preference:
    1. Use existing valid pubchem_cid if present
    2. Lookup by InChIKey if available and valid
    3. Lookup by SMILES if available and valid
    4. Lookup by drug name (perturbagen)
    5. Use manual mapping from pubchem_mapping_l1000()
    
    Uses universal cache keys (pert_id or perturbagen) to avoid redundant lookups
    when the same compound is identified by different methods.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to add pubchem_cid column to
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs.
        Will be updated with new entries during processing.
    pert_id_col : Optional[str], default='pert_id'
        Column name for perturbation ID to use as universal cache key.
        If None, uses drug_col (perturbagen) as universal cache key.
    drug_col : str, default='perturbagen'
        Column name containing drug names (used as universal cache key if pert_id_col is None)
    pubchem_cid_col : str, default='pubchem_cid'
        Column name for PubChem CID (will be created/updated)
    inchikey_col : str, default='inchi_key'
        Column name for InChIKey
    smiles_col : str, default='canonical_smiles'
        Column name for SMILES
    cache_path : Optional[str], default=None
        Path to JSON file for persistent cache storage.
        If provided, cache will be loaded from this file at start and saved periodically.
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with updated pubchem_cid column
    """
    df = df.copy()

    if df is None:
        raise Exception("The pseudobulk dataset is empty")
    
    
    # Load cache from file if path is provided
    if cache_path:
        file_cache = load_cache_from_json(cache_path)
        # Merge file cache with provided cache (provided cache takes precedence)
        cache.update(file_cache)
    
    # Initialize pubchem_cid column if it doesn't exist
    if pubchem_cid_col not in df.columns:
        df[pubchem_cid_col] = None
    
    # Get manual mappings
    sm2pubchem = pubchem_mapping_l1000()
    manual_mapping = sm2pubchem.get('l1000', {})
    
    # Process each row to determine best CID source
    cids = []
    iteration_count = 0
    n_compounds = len(df)
    logger.info(f"Processing {n_compounds} compounds")
    for idx, row in df.iterrows():
        iteration_count += 1
        cid = None
        
        # Determine universal cache key (pert_id or perturbagen)
        if pert_id_col and pert_id_col in row and pd.notna(row[pert_id_col]):
            universal_key = str(row[pert_id_col]).strip()
        elif drug_col in row and pd.notna(row[drug_col]):
            universal_key = str(row[drug_col]).strip()
        else:
            universal_key = None
        
        # Check universal cache first
        if universal_key and universal_key in cache:
            cid = cache[universal_key]
        
        # Strategy 1: Use existing valid pubchem_cid
        if cid is None and pubchem_cid_col in row and is_valid_pubchem_cid(row[pubchem_cid_col]):
            cid = int(row[pubchem_cid_col])
            # Store in universal cache
            if universal_key:
                cache[universal_key] = cid
        
        # Strategy 2: Lookup by InChIKey (if CID not found yet)
        if cid is None and inchikey_col in row and pd.notna(row[inchikey_col]):
            inchikey = str(row[inchikey_col]).strip()
            if is_valid_inchikey(inchikey):
                cid = get_pubchem_cid_by_inchikey(inchikey, cache, universal_key)
        
        # Strategy 3: Lookup by SMILES (if CID not found yet)
        if cid is None and smiles_col in row and pd.notna(row[smiles_col]):
            smiles = str(row[smiles_col]).strip()
            if is_valid_smiles(smiles):
                cid = get_pubchem_cid_by_smiles(smiles, cache, universal_key)
        
        # Strategy 4: Lookup by drug name (if CID not found yet)
        if cid is None and drug_col in row and pd.notna(row[drug_col]):
            drug_name = str(row[drug_col]).strip()
            cid = get_pubchem_cid_by_name(drug_name, cache, universal_key)
        
        # Strategy 5: Manual mapping (if CID not found yet)
        if cid is None and drug_col in row and pd.notna(row[drug_col]):
            drug_name = str(row[drug_col]).strip()
            if drug_name in manual_mapping:
                cid = manual_mapping[drug_name]
                # Store in universal cache
                if universal_key:
                    cache[universal_key] = cid
        
        cids.append(cid)
        
        # Log progress every 50 compounds
        if iteration_count % 50 == 0:
            n_mapped_so_far = sum(1 for c in cids if c is not None)
            logger.info(f"Processed {iteration_count}/{n_compounds} compounds ({n_mapped_so_far} mapped so far)")
        
        # Save cache every 10 iterations if cache_path is provided
        if cache_path and iteration_count % 500 == 0:
            save_cache_to_json(cache, cache_path)
    
    # Final save of cache if cache_path is provided
    if cache_path:
        save_cache_to_json(cache, cache_path)
    
    # Update pubchem_cid column
    df_updated = df.copy()
    df_updated[pubchem_cid_col] = cids
    
    n_mapped = df_updated[pubchem_cid_col].notna().sum()
    logger.info(f"Mapped {n_mapped} out of {len(df)} compounds to PubChem CIDs")
    
    return df_updated

In [37]:
"""
Dataset-specific processing functions for L1000 Level 3 assembly.

This module contains functions to assemble L1000 Level 3 data from GCTX files
into a standardized AnnData format matching the pseudobulk schema.
"""
from __future__ import annotations

from typing import Optional, Iterable
from pathlib import Path
import gzip
import shutil
import subprocess
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

# Try to import cmapPy for GCTX parsing
try:
    from cmapPy.pandasGEXpress import parse
    import h5py
    HAS_CMAPPY = True
except ImportError:
    HAS_CMAPPY = False
    logger.warning("cmapPy not available. GCTX parsing will not work.")

# Global variables for caching GCTX metadata
_GCTX_COL_IDS = None
_GCTX_COL_ID_MAP = None

def _read_table(path: Path, **kwargs) -> pd.DataFrame:
    """
    Read a table file, handling gzipped files automatically.
    
    Attempts to read the file at the specified path. If not found, looks for
    a gzipped version (.gz) and reads it instead.
    
    Parameters
    ----------
    path : Path
        Path to the table file (CSV format)
    **kwargs
        Additional arguments passed to pd.read_csv()
        
    Returns
    -------
    pd.DataFrame
        Loaded table data
        
    Raises
    ------
    FileNotFoundError
        If neither the file nor its gzipped version exists
    """
    if not path.exists():
        gz = path.with_suffix(path.suffix + ".gz")
        if gz.exists():
            import gzip
            with gzip.open(gz, "rt") as fh:
                df = pd.read_csv(fh, **kwargs)
        else:
            raise FileNotFoundError(path)
    else:
        df = pd.read_csv(path, **kwargs)
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.lower()
    )
    return df


def get_download_manifest(data_root: Path, dataset: str = "l1000_phase1") -> pd.DataFrame:
    """
    Get manifest of L1000 Level 3 files to download.
    
    Returns a DataFrame with download information for all required L1000 Level 3
    files from GEO and CLUE resources.
    
    Parameters
    ----------
    data_root : Path
        Root directory where files will be downloaded
        
    Returns
    -------
    pd.DataFrame
        Download manifest with columns: file, kind, size, url, path, notes, curl_example
    """
    if dataset == "l1000_phase1":
        download_manifest = [
            {
                "file": "GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx.gz",
                "kind": "expression",
                "size": "...",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx.gz",
                "notes": "Processed expression"
            },
            {
                "file": "GSE92742_Broad_LINCS_sig_info.txt.gz", 
                "kind": "metadata", 
                "size": "~10.6 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_sig_info.txt.gz",
                "notes": "Signatures information"
            },
            {
                "file": "GSE92742_Broad_LINCS_inst_info.txt.gz",
                "kind": "metadata",
                "size": "~150 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_inst_info.txt.gz",
                "notes": "Instance-level annotations"
            },
            {
                "file": "GSE92742_Broad_LINCS_cell_info.txt.gz",
                "kind": "metadata",
                "size": "<10 KB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_cell_info.txt.gz",
                "notes": "Cell line annotations"
            },
            {
                "file": "GSE92742_Broad_LINCS_pert_info.txt.gz",
                "kind": "metadata",
                "size": "~5 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_pert_info.txt.gz",
                "notes": "Perturbagen annotations"
            },
            {
                "file": "GSE92742_Broad_LINCS_gene_info.txt.gz",
                "kind": "metadata",
                "size": "<1 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_gene_info.txt.gz",
                "notes": "Gene annotations"
            },
            {
                "file": "geneinfo_beta.txt",
                "kind": "metadata",
                "size": "<1 MB",
                "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/geneinfo_beta.txt",
                "notes": "Beta gene info with Ensembl IDs"
            },
            {
                "file": "siginfo_beta.txt",
                "kind": "metadata",
                "size": "1.09 MB",
                "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/siginfo_beta.txt",
                "notes": "siginfo beta"
            },
            {
                "file": "cellinfo_beta.txt",
                "kind": "metadata",
                "size": "<100 KB",
                "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/cellinfo_beta.txt",
                "notes": "Beta cell info with Cellosaurus IDs"
            }
        ]
    elif dataset == "l1000_phase2":
        download_manifest = [
                {
                "file": "GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx.gz",
                "kind": "expression",
                "size": "12.6 GB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx.gz",
                "notes": "Raw epsilon (landmark genes)"
            },
               {
                "file": "GSE70138_Broad_LINCS_sig_info_2017-03-06.txt.gz", 
                "kind": "metadata", 
                "size": "~1.9 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_sig_info_2017-03-06.txt.gz",
                "notes": "Signatures information"
            },
            {
                "file": "GSE70138_Broad_LINCS_inst_info_2017-03-06.txt.gz",
                "kind": "metadata",
                "size": "~150 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_inst_info_2017-03-06.txt.gz",
                "notes": "Instance-level annotations"
            },
            {
                "file": "GSE70138_Broad_LINCS_cell_info_2017-04-28.txt.gz",
                "kind": "metadata",
                "size": "<10 KB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_cell_info_2017-04-28.txt.gz",
                "notes": "Cell line annotations"
            },
            {
                "file": "GSE70138_Broad_LINCS_pert_info.txt.gz",
                "kind": "metadata",
                "size": "~5 MB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_pert_info.txt.gz",
                "notes": "Perturbagen metadata"
            },
            {
                "file": "GSE70138_Broad_LINCS_gene_info_2017-03-06.txt.gz",
                "kind": "metadata",
                "size": "~210 KB",
                "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE70nnn/GSE70138/suppl/GSE70138_Broad_LINCS_gene_info_2017-03-06.txt.gz",
                "notes": "Landmark gene annotations"
            },
            {
                "file": "siginfo_beta.txt",
                "kind": "metadata",
                "size": "1.09 MB",
                "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/siginfo_beta.txt",
                "notes": "siginfo beta"
            },
            {
                "file": "geneinfo_beta.txt",
                "kind": "metadata",
                "size": "1.09 MB",
                "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/geneinfo_beta.txt",
                "notes": "2020 CLUE gene dictionary (for Ensembl IDs)"
            },
            {
                "file": "cellinfo_beta.txt",
                "kind": "metadata",
                "size": "<100 KB",
                "url": "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/cellinfo_beta.txt",
                "notes": "2020 CLUE Cell line annotations (for Cellosaurus IDs)"
            }
        ]
    else:
        raise ValueError(f"Invalid dataset: {dataset}")
    
    manifest_df = pd.DataFrame(download_manifest)
    manifest_df["path"] = manifest_df["file"].apply(lambda f: data_root / f)
    manifest_df["curl_example"] = manifest_df.apply(
        lambda row: f"curl -L '{row['url']}' -o {row['path']}",
        axis=1
    )
    
    return manifest_df


def download_l1000_files(data_root: Optional[str] = None, dataset: str = "l1000_phase1", skip_existing: bool = True) -> None:
    """
    Download L1000 Level 3 data files from GEO and CLUE.
    
    Downloads all required L1000 Level 3 files including:
    - GCTX expression file (48.8 GB)
    - Instance metadata
    - Cell line metadata
    - Perturbagen metadata
    - Gene annotations
    
    Parameters
    ----------
    data_root : str, optional
        Root directory for data files. If None, uses default from define_paths()
    skip_existing : bool, default=True
        If True, skips downloading files that already exist
        
    Raises
    ------
    subprocess.CalledProcessError
        If download command fails
    """
    if data_root is None:
        paths = define_paths(dataset=dataset)
        data_root = Path(paths["level5_gctx"]).parent
    else:
        data_root = Path(data_root)
    
    data_root.mkdir(parents=True, exist_ok=True)
    
    manifest_df = get_download_manifest(data_root, dataset=dataset)
    
    logger.info(f"Downloading L1000 Level 3 files to: {data_root}")
    
    for _, row in manifest_df.iterrows():
        file_path = row["path"]
        
        if skip_existing and file_path.exists():
            logger.info(f"  {row['file']} already exists, skipping")
            continue
        
        logger.info(f"  Downloading {row['file']} ({row['size']})...")
        cmd = f"curl -L '{row['url']}' -o {file_path}"
        
        try:
            subprocess.run(cmd, shell=True, check=True)
            logger.info(f"    Downloaded {row['file']}")
        except subprocess.CalledProcessError as e:
            logger.error(f"    Failed to download {row['file']}: {e}")
            raise
    
    logger.info("All downloads complete")


def decompress_l1000_files(data_root: Optional[str] = None, dataset: str = "l1000_phase1") -> None:
    """
    Decompress gzipped L1000 data files.
    
    Decompresses all .gz files in the data directory, including the large GCTX
    expression file. The GCTX decompression may take several minutes.
    
    Parameters
    ----------
    data_root : str, optional
        Root directory containing compressed files. If None, uses default from define_paths()
    dataset : str, default="l1000_phase1"
        Dataset name: "l1000_phase1" or "l1000_phase2"
        
    Raises
    ------
    FileNotFoundError
        If compressed files are not found
    """
    if data_root is None:
        paths = define_paths(dataset=dataset)
        data_root = Path(paths["level5_gctx"]).parent
    else:
        data_root = Path(data_root)
    
    paths = define_paths(str(data_root), dataset=dataset)
    to_decompress = []
    already_done = []
    
    for key, path in paths.items():
        if key.endswith("_gz"):
            continue
        
        path = Path(path)
        if not path.suffix == ".gz":
            gz_candidate = path.with_suffix(path.suffix + ".gz")
            if gz_candidate.exists() and not path.exists():
                to_decompress.append((key, gz_candidate, path))
            elif path.exists():
                already_done.append((key, path))
    
    if already_done:
        logger.info(f"{len(already_done)} file(s) already decompressed")
        for key, path in already_done:
            logger.info(f"  - {key}: {path.name}")
    
    if to_decompress:
        logger.info(f"Decompressing {len(to_decompress)} file(s)...")
        for key, gz_path, target_path in to_decompress:
            logger.info(f"  - {key}: {gz_path.name} -> {target_path.name}")
            if "gctx" in key.lower():
                logger.info("    (GCTX is large, this may take a few minutes...)")
            
            with gzip.open(gz_path, "rb") as src, open(target_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
            logger.info(f"    Done")
        
        logger.info(f"All {len(to_decompress)} file(s) successfully decompressed")
    else:
        if not already_done:
            logger.warning("No compressed files found. Download them first using download_l1000_files()")
        else:
            logger.info("All files are already decompressed")


def check_l1000_files(data_root: Optional[str] = None, dataset: str = "l1000_phase1") -> dict:
    """
    Check status of L1000 data files.
    
    Checks which files are missing, compressed, or ready to use.
    
    Parameters
    ----------
    data_root : str, optional
        Root directory to check. If None, uses default from define_paths()
    dataset : str, default="l1000_phase1"
        Dataset name: "l1000_phase1" or "l1000_phase2"
        
    Returns
    -------
    dict
        Dictionary with keys 'missing', 'compressed', 'ready' containing lists of
        (key, path) tuples for each category
    """
    if data_root is None:
        paths = define_paths(dataset=dataset)
        data_root = Path(paths["sig"]).parent
    else:
        data_root = Path(data_root)
    
    paths = define_paths(str(data_root), dataset=dataset)
    
    missing = []
    compressed = []
    ready = []
    
    for key, path in paths.items():
        if key.endswith("_gz"):
            continue
        
        # Skip pubchem cache JSON files
        path_str = str(path)
        if "pubchem_cache" in key.lower() and path_str.endswith(".json"):
            continue
        
        path = Path(path)
        if path.suffix == ".gz":
            if not path.exists():
                missing.append((key, path))
        else:
            gz_candidate = path.with_suffix(path.suffix + ".gz")
            if path.exists():
                ready.append((key, path))
            elif gz_candidate.exists():
                compressed.append((key, gz_candidate))
            else:
                missing.append((key, path))
    
    # Log status
    if missing:
        logger.warning(f"{len(missing)} file(s) missing:")
        for key, path in missing:
            logger.warning(f"  - {key}: {path.name}")
        logger.info("  Run download_l1000_files() to download them")
    else:
        logger.info("All required files are present")
    
    if compressed:
        logger.warning(f"{len(compressed)} file(s) still compressed:")
        for key, path in compressed:
            logger.warning(f"  - {key}: {path.name}")
        logger.info("  Run decompress_l1000_files() to extract them")
    else:
        if not missing:
            logger.info("All files are uncompressed and ready to use")
    
    return {
        "missing": missing,
        "compressed": compressed,
        "ready": ready
    }


def standardize_sig(sig: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize signature metadata.
    
    This function standardizes the signature ID column to 'lincs_sig_id' and adds plate
    information if missing by parsing the signature ID.
    
    Parameters
    ----------
    sig : pd.DataFrame
        Instance metadata with sig_id, sample_id, or distil_id column
        
    Returns
    -------
    pd.DataFrame
        Standardized dataframe with 'lincs_sig_id' as index and column, and 'det_plate' column added if missing
    """
    df = sig.copy()
    
    # Find ID column
    for id_col in ["sig_id", "sample_id", "distil_id"]:
        if id_col in df.columns:
            df = df.rename(columns={id_col: "lincs_sig_id"})
            break
    else:
        raise KeyError("sig info missing sig_id/sample_id/distil_id")
    
    # Add plate info if missing
    #if 'det_plate' not in df.columns:
    #    df['det_plate'] = df['lincs_sig_id'].str.split(':').str[0]
    
    return df.set_index("lincs_sig_id", drop=False)





def process_cellinfo(cellinfo: pd.DataFrame,
                     cellinfo_extra: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """
    Process cell info with optional extra metadata.
    
    Merges base cell info with extra metadata with Cellosaurus IDs.
    
    Parameters
    ----------
    cellinfo : pd.DataFrame
        Base cell line information with cell_id column
    cellinfo_extra : pd.DataFrame, optional
        Extra cell line metadata with Cellosaurus IDs
        
    Returns
    -------
    pd.DataFrame
        Processed cell info indexed by cell_id with prefixed column names
    """
    df = cellinfo.copy()
    cellinfo_id = 'cell_id'
    if cellinfo_extra is not None and not cellinfo_extra.empty:
        df_extra = cellinfo_extra.copy()
        df_extra[cellinfo_id] = df_extra['cell_iname'].str.replace('_', '.')
        df = pd.concat([df, df_extra])[df.columns].drop_duplicates(cellinfo_id).copy()
        df = df.merge(df_extra[[cellinfo_id, 'cellosaurus_id']], on=cellinfo_id, how='left')
        df[cellinfo_id + '_mixed'] = df['cellosaurus_id'].fillna(df[cellinfo_id])
    rename_map = {col: f"cellinfo_{col}" for col in df.columns if col != "cell_id"}
    return df.rename(columns=rename_map).set_index("cell_id")

def process_pert_metadata(pert: pd.DataFrame, sig: pd.DataFrame) -> pd.DataFrame:
    """
    Process perturbation metadata.
    
    Parameters
    ----------
    pert : pd.DataFrame
        Perturbation metadata
    sig : pd.DataFrame
        Instance metadata
        
    Returns
    -------
    pd.DataFrame
        Processed perturbation metadata
    """
    pert = pert.copy()
    sig = sig.copy()

    if not 'pubchem_cid' in pert.columns:
        pert['pubchem_cid'] = None

    return pd.concat([pert, sig]).drop_duplicates('pert_id', keep='first')[pert.columns]


def standardize_dose(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize dose units to micromolar.
    
    Converts dose values from various units (um, nm, mm) to micromolar (uM).
    Handles unit variations including µm and full names.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with 'pert_dose' and 'pert_dose_unit' columns
        
    Returns
    -------
    pd.DataFrame
        Input dataframe with additional 'pert_dose_um' column in micromolar
    """
    out = df.copy()
    dose = pd.to_numeric(out.get("pert_dose"), errors="coerce")
    unit = out.get("pert_dose_unit").astype("string").str.strip().str.lower()
    dose_um = pd.Series(np.nan, index=out.index, dtype=float)
    mask_um = unit.isin({"um", "µm", "micromolar"})
    mask_nm = unit.isin({"nm", "nanomolar"})
    mask_mm = unit.isin({"mm", "millimolar"})
    dose_um[mask_um] = dose[mask_um]
    dose_um[mask_nm] = dose[mask_nm] / 1000.0
    dose_um[mask_mm] = dose[mask_mm] * 1000.0
    out["pert_dose_um"] = dose_um
    return out


def standardize_time(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize time units to hours.
    
    Converts time values from various units (hours, days, minutes) to hours.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with 'pert_time' and 'pert_time_unit' columns
        
    Returns
    -------
    pd.DataFrame
        Input dataframe with additional 'pert_time_h' column in hours
    """
    out = df.copy()
    time_val = pd.to_numeric(out.get("pert_time"), errors="coerce")
    time_unit = out.get("pert_time_unit").astype("string").str.strip().str.lower()
    hours = pd.Series(np.nan, index=out.index, dtype=float)
    hours[time_unit.isin({"h", "hr", "hrs", "hour", "hours"})] = time_val[time_unit.isin({"h", "hr", "hrs", "hour", "hours"})]
    hours[time_unit.isin({"d", "day", "days"})] = time_val[time_unit.isin({"d", "day", "days"})] * 24.0
    hours[time_unit.isin({"m", "min", "mins", "minute", "minutes"})] = time_val[time_unit.isin({"m", "min", "mins", "minute", "minutes"})] / 60.0
    out["pert_time_h"] = hours
    return out


def build_development_stage(row: pd.Series, 
                            col: str = "cellinfo_donor_age") -> str:
    """
    Build development stage label from donor age.
    
    Creates a standardized development stage label in the format
    "{age}-year-old stage" from numeric age values.
    
    Parameters
    ----------
    row : pd.Series
        Row of signature metadata
    col : str, default="cellinfo_donor_age"
        Column name containing age information
        
    Returns
    -------
    str
        Development stage label (e.g., "69-year-old stage") or "unknown"
    """
    if col in row.index and pd.notna(row[col]):
        try:
            age = float(row[col])
            if np.isfinite(age) and age > 0:
                return f"{int(age)}-year-old stage"
        except ValueError:
            pass
    return "unknown"


def process_gene_annotations(geneinfo: pd.DataFrame,
                            geneinfo_beta: Optional[pd.DataFrame] = None,
                            full_gene_matrix: bool = False) -> pd.DataFrame:
    """
    Process gene annotations from L1000 gene info tables.
    
    Parameters
    ----------
    geneinfo : pd.DataFrame
        Level 3 gene info table with L1000 gene annotations
    geneinfo_beta : pd.DataFrame, optional
        Beta gene info table with Ensembl ID mappings
    full_gene_matrix : bool, default=False
        If False, restrict to landmark genes only
        
    Returns
    -------
    pd.DataFrame
        Processed var dataframe with gene annotations
    """
    var = geneinfo.copy()
    
    # Rename columns to standard schema
    var = var.rename(columns={
        "pr_gene_id": "gene_id",
        "pr_gene_symbol": "symbol",
        "pr_gene_title": "gene_title",
        "pr_is_lm": "is_landmark"
    })
    
    var = var.drop_duplicates(subset=["gene_id"]).set_index("gene_id")
    var["symbol"] = var["symbol"].astype("string")
    
    # Map to Ensembl IDs if available
    if geneinfo_beta is not None and {"gene_id", "ensembl_id"}.issubset(geneinfo_beta.columns):
        sym_to_ens = (
            geneinfo_beta[["gene_id", "ensembl_id"]]
            .dropna()
            .drop_duplicates(subset=["gene_id"])
            .set_index("gene_id")
            ["ensembl_id"]
        )
        var["ensembl_id"] = var.index.map(sym_to_ens)
    else:
        var["ensembl_id"] = var.index.astype("string")
    
    # Fill missing Ensembl IDs with gene_id
    var["ensembl_id"] = var["ensembl_id"].fillna(var.index.to_series().astype("string"))
    var = var[["symbol", "ensembl_id", "is_landmark"]]
    
    # Filter to landmark genes if requested
    if not full_gene_matrix:
        landmark_mask = var["is_landmark"].fillna(0).astype(int)
        var = var.loc[landmark_mask == 1]
        logger.info(f'  Restricting to {len(var):,} landmark genes')
    else:
        logger.info(f'  Using all {len(var):,} gene features')
    
    return var


def build_obs_dataframe(sig: pd.DataFrame, dataset: str = "l1000_phase1") -> pd.DataFrame:
    """
    Build standardized .obs dataframe from signature metadata.
    Sticked to https://lamin.ai/laminlabs/pertdata/transform/REAvqqdo3sbH0000
    
    Parameters
    ----------
    sig : pd.DataFrame
        Instance metadata dataframe with LINCS information
        
    Returns
    -------
    pd.DataFrame
        Standardized obs dataframe with pseudobulk schema
    """
    obs = pd.DataFrame(index=sig.index)
    
    # Copy identifier columns
    for extra_id in ("lincs_sig_id", "distil_id", "inst_id", "sample_id", "sig_id"):
        if extra_id in sig.columns:
            obs[extra_id] = sig[extra_id].astype("string")
    
    # Map perturbation types to standard schema
    PERT_TYPE_MAP = {
        "trt_cp": "compound",
        "trt_lig": "biologic",
        "trt_sh": "genetic",
        "trt_oe": "genetic",
        "trt_oe.mut": "genetic",
        "trt_xpr": "genetic",
        "ctl_vehicle": "compound",
        "trt_poscon": "compound",
        "ctl_vector": "genetic",
        "ctl_untrt": "biologic"
        
    }

    
    # Build standard obs columns
    obs["plate"] = sig.get("det_plate", None)
    obs["well"] = sig.get("rna_well", sig.get("det_well", None))
    obs["cell_type"] = sig.get("cellinfo_cell_id_mixed", sig.get("cell_id", None)).fillna(sig.get("cell_id", None))
    obs["perturbagen"] = sig.get("pert_iname", None)
    obs["pert_type"] = sig["pert_type"].map(PERT_TYPE_MAP)
    obs["is_control"] = sig["pert_type"].str.startswith("ctl")
    obs["pert_dose_uM"] = sig["pert_dose_um"].astype(float)
    obs.loc[(obs['perturbagen']=='DMSO') & (obs['is_control']==True), 'pert_dose_uM'] = 0
    obs['pert_dose'] = sig['pert_dose'].astype(str) + ' ' + sig['pert_dose_unit'].astype(str)
    obs["pert_time_h"] = sig["pert_time_h"].astype(float)
    obs["suspension_type"] = "cell"
    obs["tissue"] = sig.get("cellinfo_primary_site", "unknown")
    obs["tissue_type"] = "cell culture"
    obs["disease"] = sig.get("cellinfo_subtype", "unknown")
    obs["library"] = None
    obs["stimulation"] = None
    obs["guide"] = None

    if dataset == "l1000_phase1":
        obs["dataset"] = "LINCS_phase1_level3_epsilon"
    elif dataset == "l1000_phase2":
        obs["dataset"] = "LINCS_phase2_level3"
    else:
        raise ValueError(f"Invalid dataset: {dataset}")

    obs["assay"] = "L1000 mRNA profiling assay"
    obs["development_stage"] = sig.apply(build_development_stage, axis=1)
    obs["organism"] = "human"
    obs["sex"] = sig["cellinfo_donor_sex"].map({"M": "male", "F": "female"})
    obs["self_reported_ethnicity"] = sig.get("cellinfo_donor_ethnicity", "unknown")
    if 'pubchem_cid' in sig.columns:
        sig['pubchem_cid'] = pd.to_numeric(sig['pubchem_cid'], errors='coerce').fillna(-666).astype('int64')
    obs["pubchem_cid"] = sig.get("pubchem_cid", None)
    obs["psbulk_cells"] = None
    obs["psbulk_counts"] = None
    
    # Add metadata columns
    obs["lincs_sig_id"] = sig.index.astype("string")
    obs["source_gctx"] = sig["source_gctx"].astype("string")
    
    # Create composite sample_id
    obs["sample_id"] = (
        obs["plate"].astype(str).str.replace(" ", "", regex=False) + "_" +
        obs["well"].astype(str).str.replace(" ", "", regex=False) + "_" +
        obs["perturbagen"].astype(str).str.replace(" ", "_", regex=False) + "_" +
        obs["cell_type"].astype(str).str.replace(" ", "_", regex=False)
    )

    obs["pert_type_init"] = sig["pert_type_pert"].copy()
    # Clean up missing values and duplicates
    obs = obs.replace({-666: None, '-666': None, 'None': None, 'nan': None, '<NA>': None})
    #obs = obs[~obs["sample_id"].duplicated(keep="first")]
    obs = obs.set_index("sig_id", drop=True)
    obs = materialize_string_columns(obs)
    return obs


def enforce_obs_schema(obs: pd.DataFrame) -> pd.DataFrame:
    """
    Enforce strict obs schema on the observations dataframe.
    
    This function ensures that the obs dataframe follows the strict 24-column
    schema defined in OBS_SCHEMA. It:
    1. Adds missing columns as NaN
    2. Casts columns to the correct dtypes (category, float64, int64)
    3. Fills missing values appropriately (unknown for certain category columns, -666 for int64)
    4. Selects only the columns in the schema
    5. Materializes string columns for AnnData compatibility
    
    Parameters
    ----------
    obs : pd.DataFrame
        Observations dataframe to enforce schema on
        
    Returns
    -------
    pd.DataFrame
        Observations dataframe conforming to the strict schema
    """
    # Columns that should be filled with "unknown" instead of NaN
    cols_fillna_unknown = [
        'tissue',
        'tissue_type',
        'disease',
        'development_stage',
        'sex',
        'self_reported_ethnicity'
    ]
    
    # Get obs schema
    obs_schema = define_obs_schema()
    
    # Create dtype map from schema
    dtype_map = {col: dtype for col, dtype, _ in obs_schema}
    
    # Create schema dataframe for reference
    obs_schema_df = pd.DataFrame(obs_schema, columns=["column", "dtype", "description"]).set_index("column")
    
    obs_for_schema = obs.copy()
    
    # Ensure all schema columns exist, add as NaN if missing
    for col, dtype in dtype_map.items():
        if col not in obs_for_schema.columns:
            obs_for_schema[col] = np.nan
        
        # Cast to appropriate dtype
        if dtype == "category":
            if col in cols_fillna_unknown:
                obs_for_schema[col] = (
                    obs_for_schema[col]
                    .fillna("unknown")
                    .astype("string")
                    .astype(object)
                    .astype("category")
                )
            else:
                obs_for_schema[col] = (
                    obs_for_schema[col]
                    .astype(object)
                    .astype("category")
                )
        elif dtype == "float64":
            obs_for_schema[col] = pd.to_numeric(obs_for_schema[col], errors="coerce")
        elif dtype == "int64":
            obs_for_schema[col] = (
                pd.to_numeric(obs_for_schema[col], errors="coerce")
                .fillna(-666)
                .astype("int64")
            )
    
    # Select only columns in schema
    obs_for_schema = obs_for_schema[obs_schema_df.index.tolist()]
    
    # Materialize string columns for AnnData compatibility
    obs_for_schema = materialize_string_columns(obs_for_schema)
    
    return obs_for_schema


def build_config(config: Optional[dict] = None) -> dict:
    """
    Build L1000 configuration by merging defaults with provided config.
    
    Parameters
    ----------
    config : dict, optional
        User-provided configuration parameters
        
    Returns
    -------
    dict
        Complete configuration with defaults filled in. Keys include:
        - perturbation_types_to_keep: set of perturbation types to include
        - control: set of control perturbagen names
        - full_gene_matrix: bool, whether to use full gene matrix
        - subsampling: bool, whether to use subsampling of the dataset
        - annotate_pubchem: bool, whether to annotate perturbations with PubChem CIDs
        - download_if_missing: bool, whether to auto-download missing files (default: True)
    """
    # Set defaults
    if config is None:
        config = {}
    
    # Configuration defaults
    default_config = {
        "perturbation_types_to_keep": {"trt_cp", "ctl_vehicle"},
        "control": {"DMSO"},
        "full_gene_matrix": False,
        "subsampling": False,
        "annotate_pubchem": False,
        "download_if_missing": True,
    }
    
    # Merge with provided config
    return {**default_config, **config}


def define_obs_schema() -> list:
    """
    Define the strict obs schema for L1000 pseudobulk data.
    
    Returns
    -------
    list
        List of tuples defining the schema: (column_name, dtype, description)
    """
    return [
        ("sample_id", "category", "ID of the observation: plate + well + cell_type + perturbagen"),
        ("plate", "category", "Assay plate identifier (det_plate)"),
        ("well", "category", "Well ID on the RNA plate (rna_well)"),
        ("cell_type", "category", "Cell line / cell_id"),
        ("perturbagen", "category", "Human-readable perturbagen label"),
        ("pert_type", "category", "Perturbation class"),
        ("is_control", "category", "True/False for controls"),
        ("pert_dose_uM", "float64", "Dose in micromolar"),
        ("pert_time_h", "float64", "Exposure time in hours"),
        ("suspension_type", "category", "Growth pattern"),
        ("tissue", "category", "Primary tissue/site"),
        ("tissue_type", "category", "Sample type"),
        ("disease", "category", "Disease/subtype"),
        ("library", "category", "Library/release"),
        ("stimulation", "category", "High-level stimulus"),
        ("guide", "category", "A guide RNA directs the CRISPR system"),
        ("dataset", "category", "Dataset label"),
        ("assay", "category", "Assay label"),
        ("development_stage", "category", "Derived from donor age"),
        ("organism", "category", "Organism"),
        ("sex", "category", "Donor sex"),
        ("self_reported_ethnicity", "category", "Donor ethnicity"),
        ("pubchem_cid", "category", "PubChem CID"),
        ("psbulk_cells", "int64", "Total #cells contributing (if no info - then -666)"),
        ("psbulk_counts", "int64", "Total #counts contributing (if no info - then -666)"),
        ("distil_id", "category", "distil_id"),
        ("sig_id", "category", "signature id"),
        ("pert_type_init", "category", "initial perturbation type"),
        ("pert_dose", "category", "initial perturbation type")
    ]


def define_paths(data_root: Optional[str] = None, dataset: str = "l1000_phase1") -> dict:
    """
    Define file paths for L1000 Level 3 data.
    
    Parameters
    ----------
    data_root : str, optional
        Root directory containing L1000 data files.
        Defaults to './lincs_data'
        
    Returns
    -------
    dict
        Dictionary mapping file identifiers to Path objects
    """
    if data_root is None:
        data_root = './lincs_data'
    
    data_root = Path(data_root)
    processed_dir = data_root / "processed"
    processed_dir.mkdir(exist_ok=True)
    
    if dataset == "l1000_phase1":
        return {
                "level5_gctx": data_root / "GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx",
                "level5_gctx_gz": data_root / "GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx.gz",
                "instinfo": data_root / "GSE92742_Broad_LINCS_inst_info.txt",
                "siginfo": data_root / "GSE92742_Broad_LINCS_sig_info.txt",
                "cellinfo": data_root / "GSE92742_Broad_LINCS_cell_info.txt",
                "pert_info": data_root / "GSE92742_Broad_LINCS_pert_info.txt",
                "geneinfo": data_root / "GSE92742_Broad_LINCS_gene_info.txt",
                "geneinfo_beta": data_root / "geneinfo_beta.txt",
                "siginfo_beta": data_root / "siginfo_beta.txt",
                "cellinfo_beta": data_root / "cellinfo_beta.txt",
                "pubchem_cache": processed_dir / "pubchem_cache.json",
                
        }
    elif dataset == "l1000_phase2":
        return {
                "level5_gctx": data_root / "GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx",
                "level5_gctx_gz": data_root / "GSE70138_Broad_LINCS_Level5_COMPZ_n118050x12328_2017-03-06.gctx.gz",
                "instinfo": data_root / "GSE70138_Broad_LINCS_inst_info_2017-03-06.txt",
                "siginfo": data_root / "GSE70138_Broad_LINCS_sig_info_2017-03-06.txt",
                "cellinfo": data_root / "GSE70138_Broad_LINCS_cell_info_2017-04-28.txt",
                "pert_info": data_root / "GSE70138_Broad_LINCS_pert_info.txt",
                "geneinfo": data_root / "GSE70138_Broad_LINCS_gene_info_2017-03-06.txt",
                "geneinfo_beta": data_root / "geneinfo_beta.txt",
                "siginfo_beta": data_root / "siginfo_beta.txt",
                "cellinfo_beta": data_root / "cellinfo_beta.txt",
                "pubchem_cache": processed_dir / "pubchem_cache.json",
        }
    else:
        raise ValueError(f"Invalid dataset: {dataset}")

def add_alternative_identifiers(sig_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize signature metadata and add alternative identifier columns.
    
    This function standardizes the signature ID column and preserves alternative
    identifiers (distil_id, sig_id, sample_id, sig_id) for cross-referencing.
    
    Parameters
    ----------
    sig_raw : pd.DataFrame
        Raw signature/sample metadata
        
    Returns
    -------
    pd.DataFrame
        Instance metadata with standardized lincs_sig_id index
        and additional identifier columns
    """
    logger.info('  Processing signature metadata')
    sig = standardize_sig(sig_raw)
    
    # Keep alternative identifier columns
    sig_raw_tmp = sig_raw.copy()
    if 'lincs_sig_id' not in sig_raw_tmp.columns:
        if 'sample_id' in sig_raw_tmp.columns:
            sig_raw_tmp['lincs_sig_id'] = sig_raw_tmp['sample_id']
        elif 'sig_id' in sig_raw_tmp.columns:
            sig_raw_tmp['lincs_sig_id'] = sig_raw_tmp['sig_id']
        else:
            raise ValueError("signature metadata must have lincs_sig_id, sample_id, or sig_id column")
    
    sig_raw_indexed = sig_raw_tmp.set_index("lincs_sig_id", drop=False)
    
    for extra_id in ("distil_id", "sig_id", "sample_id", "sig_id"):
        if extra_id in sig_raw_indexed.columns:
            sig[extra_id] = sig_raw_indexed.loc[sig.index, extra_id].astype("string")
    
    return sig


def annotate_pubchem_cids(df: pd.DataFrame, paths: dict, config: dict = None) -> pd.DataFrame:
    """
    Annotate metadata containing perturbation information with PubChem CIDs.
    
    This function adds or updates PubChem CID information for perturbations
    using multiple lookup strategies (InChIKey, SMILES, drug name) with
    persistent caching to avoid redundant API calls.
    
    Only compound perturbations (trt_cp and ctl_vehicle) are annotated,
    as other perturbation types (shRNA, CRISPR, etc.) don't have PubChem CIDs.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Metadata dataframe containing perturbation information (e.g., signature metadata
        enriched with perturbation data via merge). Must contain columns: pert_id, pert_type,
        pert_iname, and optionally inchi_key, canonical_smiles.
    paths : dict
        Dictionary of file paths from define_paths(), must include 'pubchem_cache'
        
    Returns:
    --------
    pd.DataFrame
        Metadata dataframe with updated pubchem_cid column
    """
    def standardize_pubchem_cid(cid):
        try:
            if pd.isna(cid):
                return None
            cid_int = int(cid)
            return cid_int if cid_int > 0 else None
        except Exception as e:
            logger.warning(f"Error standardizing pubchem_cid: {e}")
            return None

    # Filter to only compound perturbations (trt_cp and ctl_vehicle)
    # Other types (shRNA, CRISPR, etc.) don't have PubChem CIDs
    df = df.copy()

    if not 'pubchem_cid' in df.columns:
        df['pubchem_cid'] = None

    df['pubchem_cid'] = df['pubchem_cid'].apply(standardize_pubchem_cid)
    df['pubchem_cid'] = pd.to_numeric(df['pubchem_cid'], errors='coerce').fillna(-666).astype('int64')

    compound_types = {"trt_cp", "ctl_vehicle"}
    df_compounds = df[df["pert_type"].isin(compound_types)].copy()

    if len(df_compounds) == 0:
        logger.warning("No compound perturbations found to annotate")
        return df

    if config is not None and config.get("subsampling"):
        df_compounds = df_compounds.sample(n=1000, random_state=0)
    
    
    
    
    pubchem_cache = {}
    cache_path = str(paths["pubchem_cache"])
    df_compounds = add_pubchem_cids(
        df_compounds, 
        cache=pubchem_cache, 
        pert_id_col='pert_id',
        drug_col='pert_iname',
        cache_path=cache_path
    )
    
    df_compounds['pubchem_cid'] = pd.to_numeric(df_compounds['pubchem_cid'], errors='coerce').fillna(-666).astype("int64")
    
    # Update the original df with annotated CIDs
    df.loc[df_compounds.index, "pubchem_cid"] = df_compounds["pubchem_cid"]
    return df


def enrich_sig_metadata(sig: pd.DataFrame, 
                            cellinfo: pd.DataFrame,
                            pert_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Enrich instance metadata by merging with cell and perturbation info.
    
    Parameters
    ----------
    sig : pd.DataFrame
        Processed signature metadata
    cellinfo : pd.DataFrame
        Cell line information
    pert_raw : pd.DataFrame
        Perturbation information
        
    Returns
    -------
    pd.DataFrame
        Enriched signature metadata with merged information
    """
    pert_cols = ["pert_id", "pert_iname", "pert_type", "pubchem_cid"]
    sig = sig.merge(cellinfo, how="left", left_on="cell_id", right_index=True)    
    sig = sig.merge(pert_raw[pert_cols].drop_duplicates("pert_id"), 
                     how="left", on="pert_id", suffixes=("", "_pert"))
    
    return sig


def process_sig_metadata(sig_raw: pd.DataFrame,
                            cellinfo: pd.DataFrame,
                            pert_raw: pd.DataFrame,
                            config: dict,
                            paths: dict,
                            max_samples: int = 1000) -> pd.DataFrame:
    """
    Process signature metadata through complete pipeline.
    
    This function handles the full signature metadata processing pipeline:
    1. Add alternative identifiers (distil_id, sig_id, sample_id, sig_id)
    2. Enrich with cell and perturbation information
    3. Standardize dose units to micromolar
    4. Standardize time units to hours
    5. Apply filters (perturbation types, controls, subsampling)
    6. Add source file information
    
    Parameters
    ----------
    sig_raw : pd.DataFrame
        Raw signature/sample metadata
    cellinfo : pd.DataFrame
        Cell line information
    pert_raw : pd.DataFrame
        Perturbation information
    config : dict
        Configuration with filter settings
    paths : dict
        Dictionary of file paths
        
    Returns
    -------
    pd.DataFrame
        Fully processed and filtered signature metadata
    """
    sig = add_alternative_identifiers(sig_raw)
    sig = enrich_sig_metadata(sig, cellinfo, pert_raw)
    sig = standardize_dose(sig)
    sig = standardize_time(sig)
    
    # Apply filters
    if config["perturbation_types_to_keep"] is not None:
        sig = sig[sig["pert_type"].isin(config["perturbation_types_to_keep"])]

    if config["control"] is not None:
        sig = sig[(sig['pert_type'].str.startswith('ctl') & sig['pert_iname'].isin(config['control'])) |
                    (~sig['pert_type'].str.startswith('ctl'))]

    cell_ids_with_controls = set(sig[sig['pert_type'].str.startswith('ctl')]['cell_id'].unique())
    cell_ids_with_compounds = set(sig[~sig['pert_type'].str.startswith('ctl')]['cell_id'].unique())
    valid_cell_ids = cell_ids_with_controls & cell_ids_with_compounds
    if len(valid_cell_ids) == 0:
        raise ValueError("No cell lines found with both controls and compounds")

    sig = sig[sig['cell_id'].isin(valid_cell_ids)].copy()

    if config["subsampling"]:
        sig = sig.sample(max_samples, random_state=0)
    
    # Add source file information
    sig["source_gctx"] = str(paths["level5_gctx"])
    
    return sig


def load_metadata_tables(paths: dict) -> tuple:
    """
    Load L1000 metadata tables from files.
    
    Parameters
    ----------
    paths : dict
        Dictionary of file paths from define_paths()
        
    Returns
    -------
    tuple
        (sig_raw, cellinfo_raw, pert_raw, geneinfo, geneinfo_beta, cellinfo_beta)
        - sig_raw: Instance/sample information
        - cellinfo_raw: Cell line information
        - pert_raw: Perturbation information
        - geneinfo: Gene annotations (Level 3)
        - geneinfo_beta: Gene annotations (beta) - optional, None if not found
        - cellinfo_beta: Cell line annotations (beta) - optional, None if not found
    """
    logger.info('  Loading metadata tables')
    inst_raw = _read_table(paths["instinfo"], sep="\t", low_memory=False)
    sig_raw = _read_table(paths["siginfo"], sep="\t", low_memory=False)
    cellinfo_raw = _read_table(paths["cellinfo"], sep="\t")
    pert_raw = _read_table(paths["pert_info"], sep="\t")
    geneinfo = _read_table(paths["geneinfo"], sep="\t")
    geneinfo_beta = _read_table(paths["geneinfo_beta"], sep="\t") if paths["geneinfo_beta"].exists() else None
    siginfo_beta = _read_table(paths["siginfo_beta"], sep="\t", low_memory=False) if paths["siginfo_beta"].exists() else None
    cellinfo_beta = _read_table(paths["cellinfo_beta"], sep="\t") if paths["cellinfo_beta"].exists() else None
    
    # Log loaded table sizes
    logger.info(f"    Loaded {len(sig_raw):,} signatures")
    logger.info(f"    Loaded {len(cellinfo_raw):,} cell lines")
    logger.info(f"    Loaded {len(pert_raw):,} perturbations")
    logger.info(f"    Loaded {len(geneinfo):,} genes")
    
    return inst_raw, sig_raw, cellinfo_raw, pert_raw, geneinfo, geneinfo_beta, siginfo_beta, cellinfo_beta


def materialize_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert string columns to object type for AnnData compatibility.
    
    AnnData requires string data to be stored as object dtype rather than
    pandas StringDtype. This function converts all string columns and index
    to object type.
    
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with string-typed columns
        
    Returns
    -------
    pd.DataFrame
        Dataframe with string columns converted to object dtype
    """
    for col in df.select_dtypes(include="string").columns:
        df[col] = df[col].astype(object).where(pd.notna(df[col]), np.nan)
    
    if pd.api.types.is_string_dtype(df.index.dtype):
        df.index = df.index.astype(object)
    return df


def _get_gctx_column_ids(gctx_path: Path) -> set:
    """
    Load and cache GCTX column IDs.
    
    Parameters
    ----------
    gctx_path : Path
        Path to the GCTX file.
        
    Returns
    -------
    set
        Set of GCTX column IDs.
    """
    global _GCTX_COL_IDS, _GCTX_COL_ID_MAP
    
    if _GCTX_COL_IDS is None:
        if not gctx_path.exists():
            raise FileNotFoundError(gctx_path)
        with h5py.File(gctx_path, "r") as handle:
            root = handle["0"] if "0" in handle else handle
            ids = root["META"]["COL"]["id"][:]
        _GCTX_COL_IDS = [
            id_.decode("utf-8") if isinstance(id_, bytes) else str(id_)
            for id_ in ids
        ]
        _GCTX_COL_ID_MAP = {id_.lower(): id_ for id_ in _GCTX_COL_IDS}
        logger.info(f"  Loaded {len(_GCTX_COL_IDS):,} column IDs from {gctx_path.name}")
    return set(_GCTX_COL_IDS)


def _filter_obs_for_gctx(obs_subset: pd.DataFrame, gctx_path: Path) -> pd.DataFrame:
    """
    Filter obs to only include samples present in GCTX.
    
    Parameters
    ----------
    obs_subset : pd.DataFrame
        Observations dataframe with "gctx_id" column.
    gctx_path : Path
        Path to the GCTX file.
        
    Returns
    -------
    pd.DataFrame
        Filtered obs containing only samples present in GCTX.
        
    Raises
    ------
    ValueError
        If no samples remain after filtering.
    """
    available = _get_gctx_column_ids(gctx_path)
    mask = obs_subset["gctx_id"].isin(available)
    if not mask.all():
        dropped = int((~mask).sum())
        logger.warning(f"  Dropping {dropped} samples not present in the Level 3 matrix")
    obs_filtered = obs_subset.loc[mask].copy()
    if obs_filtered.empty:
        raise ValueError("No samples remain after intersecting with the Level 3 matrix.")
    return obs_filtered


def _normalize_gene_ids(gene_ids: Optional[Iterable[str]]) -> Optional[list]:
    """
    Normalize gene IDs to a deduplicated list.
    
    Parameters
    ----------
    gene_ids : Optional[Iterable[str]]
        Gene IDs to normalize.
        
    Returns
    -------
    Optional[list]
        Deduplicated list of gene IDs, or None if input is None.
    """
    if gene_ids is None:
        return None
    return list(dict.fromkeys(str(gid) for gid in gene_ids))


def build_obs_helpers(obs: pd.DataFrame, gctx_path: Path) -> pd.DataFrame:
    """
    Build helper dataframe with GCTX IDs for obs matching.
    
    Tries exact matching first, then falls back to case-insensitive matching.
    
    Parameters
    ----------
    obs : pd.DataFrame
        Observations dataframe with index to match against GCTX IDs.
    gctx_path : Path
        Path to the GCTX file.
        
    Returns
    -------
    pd.DataFrame
        Subset of obs with "source_gctx" and "gctx_id" columns.
        
    Raises
    ------
    ValueError
        If none of the obs IDs match the GCTX metadata.
    """
    gctx_ids = _get_gctx_column_ids(gctx_path)
    id_map = _GCTX_COL_ID_MAP or {}
    index_vals = obs.index.astype("string")
    mask = index_vals.isin(gctx_ids)
    
    if mask.any():
        matched = int(mask.sum())
        logger.info(f"  Using obs index to subset GCTX ({matched}/{len(obs)} samples match)")
        helpers = obs.loc[mask, ["source_gctx"]].copy()
        helpers["gctx_id"] = index_vals[mask]
        return helpers
    
    # Fallback: case-insensitive matching
    lower_vals = index_vals.str.lower()
    mask_lower = lower_vals.isin(id_map)
    if mask_lower.any():
        matched = int(mask_lower.sum())
        logger.info(f"  Using obs index (case-insensitive) to subset GCTX ({matched}/{len(obs)} samples match)")
        helpers = obs.loc[mask_lower, ["source_gctx"]].copy()
        helpers["gctx_id"] = lower_vals[mask_lower].map(id_map)
        return helpers
    
    raise ValueError("None of the obs ID columns match the Level 3 GCTX metadata.")


def assemble_anndata(obs: pd.DataFrame, var: pd.DataFrame, expr: pd.DataFrame) -> ad.AnnData:
    """
    Assemble final AnnData object from processed components.
    
    This function:
    1. Prepares var dataframe with Ensembl IDs as index
    2. Handles duplicate Ensembl IDs by adding suffixes
    3. Reindexes expression to match var
    4. Creates final AnnData object with sparse matrix
    
    Parameters
    ----------
    obs : pd.DataFrame
        Observations dataframe with sample metadata
    var : pd.DataFrame
        Gene annotations with gene_id as index
    expr : pd.DataFrame
        Expression matrix (genes x samples)
        
    Returns
    -------
    ad.AnnData
        Final AnnData object with standardized schema
    """
    # Prepare var dataframe
    var_df = var.reindex(expr.columns.astype(int)).copy()
    var_idx = var_df["ensembl_id"].fillna(var_df.index.to_series().astype("string"))
    var_idx = var_idx.astype(object)
    
    # Handle duplicate ensembl IDs
    duplicate_mask = var_idx.duplicated(keep=False)
    if duplicate_mask.any():
        suffix = (
            var_idx[duplicate_mask]
            .groupby(var_idx[duplicate_mask])
            .cumcount()
            .astype("string")
        )
        var_idx = var_idx.astype("string")
        var_idx[duplicate_mask] = var_idx[duplicate_mask] + "_" + suffix
    var_idx = var_idx.astype(object)
    var_df.index = var_idx
    var_df = var_df[["symbol"]]
    var_df["symbol"] = (
        var_df["symbol"]
        .replace({"": None})
        .astype(object)
        .astype("category")
    )
    var_df = materialize_string_columns(var_df)
    
    
    # Prepare final obs
    obs_final = obs.loc[expr.index].set_index('sample_id')
    
    # Create AnnData
    adata = ad.AnnData(
        X=sp.csr_matrix(expr.to_numpy(np.float32)),
        obs=obs_final,
        var=var_df
    )
    
    return adata


def load_expression(obs_helpers: pd.DataFrame, gctx_path: Path, 
                    gene_ids: Optional[Iterable[str]] = None) -> pd.DataFrame:
    """
    Load expression data from GCTX for given observations and genes.
    
    Parameters
    ----------
    obs_helpers : pd.DataFrame
        Helper dataframe with "gctx_id" column mapping to GCTX column IDs.
    gctx_path : Path
        Path to the GCTX file.
    gene_ids : Optional[Iterable[str]]
        Gene IDs to extract. If None, extracts all genes.
        
    Returns
    -------
    pd.DataFrame
        Expression dataframe with obs_helpers.index as columns and genes as rows,
        reindexed to match the requested gene_ids order.
    """
    obs_filtered = _filter_obs_for_gctx(obs_helpers, gctx_path)
    gene_ids_normalized = _normalize_gene_ids(gene_ids)
    lincs_ids = obs_filtered["gctx_id"].tolist()
    
    gctx = parse.parse(
        str(gctx_path),
        cid=lincs_ids,
        rid=gene_ids_normalized,
    )
    expr = gctx.data_df.loc[:, lincs_ids]
    expr.columns = obs_filtered.index

    expr_df = pd.DataFrame(expr.T.to_numpy(np.float32), 
                           index=expr.columns.to_numpy(str), 
                           columns=expr.index.to_numpy(str)
                           )

    expr_df = expr_df.reindex(obs_filtered.index)
    if gene_ids_normalized is not None:
        expr_df = expr_df.loc[:, gene_ids_normalized]

    expr_df = expr_df.reindex(columns=gene_ids)
    
    return expr_df


def assemble_l1000_dataset(padata: ad.AnnData,
                         data_root: Optional[str] = None,
                         config: Optional[dict] = None,
                         **kwargs) -> ad.AnnData:
    """
    Assemble L1000 Level 3 data from GCTX files into standardized AnnData format.
    
    This function assembles L1000 Level 3 data by:
    1. Checking and downloading required data files if missing
    2. Loading and standardizing metadata
    3. Optionally annotating perturbations with PubChem CIDs
    4. Building standardized .obs dataframe
    5. Processing gene annotations
    6. Assembling final AnnData object
    
    Parameters:
    -----------
    padata : ad.AnnData
        Pseudobulk AnnData object (may be empty or placeholder)
    data_root : str, optional
        Path to L1000 data root directory containing GCTX and metadata files.
        If files are missing, they will be automatically downloaded.
    config : dict, optional
        Configuration dictionary with processing parameters.
        Can include:
        - 'dataset' (str, required) to specify the dataset name (l1000_phase1 or l1000_phase2)
        - 'download_if_missing' (bool, default=True) to control automatic downloads
        - 'annotate_pubchem' (bool, default=False) to annotate perturbations with PubChem CIDs
          (may involve API calls to PubChem and can be time-consuming)
    **kwargs
        Additional processing parameters
        
    Returns:
    --------
    ad.AnnData
        Processed AnnData object with standardized schema
    """
    logger.info('Applying L1000-specific processing')

    if not HAS_CMAPPY:
        raise ImportError("cmapPy is required for L1000 processing. Install via: pip install cmapPy")
    
    # Build configuration
    CONFIG = build_config(config)
    download_if_missing = CONFIG.get("download_if_missing", True)
    
    # Define file paths
    PATHS = define_paths(data_root, dataset=CONFIG.get("dataset"))
    
    # Check data availability and download if needed
    if download_if_missing:
        logger.info("Checking data availability...")
        status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        
        if status['missing']:
            logger.info(f"Downloading {len(status['missing'])} missing file(s)...")
            download_l1000_files(data_root, dataset=CONFIG.get("dataset"), skip_existing=True)
            status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        
        
        if status['compressed']:
            logger.info(f"Decompressing {len(status['compressed'])} compressed file(s)...")
            decompress_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        
        # Verify all files are ready
        final_status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
        if final_status['missing'] or final_status['compressed']:
            raise FileNotFoundError(
                f"Required L1000 data files are still missing or compressed after download attempt. "
                f"Missing: {len(final_status['missing'])}, Compressed: {len(final_status['compressed'])}"
            )
        logger.info("All required data files are available")
    
    FULL_GENE_MATRIX = CONFIG["full_gene_matrix"]
    
    # Load metadata tables
    sig_raw, cellinfo_raw, pert_raw, geneinfo, geneinfo_beta, cellinfo_beta = load_metadata_tables(PATHS)
    
    # Process cell info
    cellinfo = process_cellinfo(cellinfo_raw, cellinfo_extra=cellinfo_beta)

    # Process perturbation metadata
    pert_raw = process_pert_metadata(pert_raw, sig_raw)

    # Annotate compounds with PubChem CIDs (optional)
    if CONFIG.get("annotate_pubchem", False):
        logger.info("Mapping compounds to PubChem CIDs")
        pert_raw = annotate_pubchem_cids(pert_raw, PATHS, config=CONFIG)
    
    # Process signature metadata
    sig = process_sig_metadata(sig_raw, cellinfo, pert_raw, CONFIG, PATHS)
    
    logger.info('  Building .obs dataframe')
    obs = build_obs_dataframe(sig, dataset=CONFIG.get("dataset"))
    
    logger.info('  Enforcing strict obs schema')
    obs_for_schema = enforce_obs_schema(obs)
    
    logger.info('  Processing gene annotations')
    var = process_gene_annotations(geneinfo, geneinfo_beta, FULL_GENE_MATRIX)
    
    logger.info('  Matching obs to GCTX column IDs')
    obs_helpers = build_obs_helpers(obs, PATHS["level5_gctx"])
    
    
    logger.info('  Extracting expression from GCTX')
    expr = load_expression(obs_helpers, PATHS["level5_gctx"], gene_ids=var.index.astype(str))
    
    logger.info('  Assembling AnnData')
    padata_processed = assemble_anndata(obs_for_schema, var, expr)
    
    logger.info(f'  Assembled AnnData: {padata_processed.n_obs:,} × {padata_processed.n_vars:,}')
    logger.info('L1000-specific processing completed')
    
    return padata_processed

In [3]:
config = {
        'data_root': './lincs_data',
        'output_file': 'level5_phase1_not_filtered.h5ad',
        'perturbation_types_to_keep': None,
        'control': None,
        'full_gene_matrix': False,
        'subsampling': False,
        'annotate_pubchem': True,
        'download_if_missing': True,
        'dataset': 'l1000_phase1',
    }

In [4]:
data_root = './lincs_data'
padata = ad.AnnData()

In [5]:
logger.info('Applying L1000-specific processing')

if not HAS_CMAPPY:
    raise ImportError("cmapPy is required for L1000 processing. Install via: pip install cmapPy")

# Build configuration
CONFIG = build_config(config)
download_if_missing = CONFIG.get("download_if_missing", True)

# Define file paths
PATHS = define_paths(data_root, dataset=CONFIG.get("dataset"))



2026-01-06 16:17:35 | [INFO] Applying L1000-specific processing


In [6]:
# Check data availability and download if needed
if download_if_missing:
    logger.info("Checking data availability...")
    status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
    
    if status['missing']:
        logger.info(f"Downloading {len(status['missing'])} missing file(s)...")
        download_l1000_files(data_root, dataset=CONFIG.get("dataset"), skip_existing=True)
        status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
    
    
    if status['compressed']:
        logger.info(f"Decompressing {len(status['compressed'])} compressed file(s)...")
        decompress_l1000_files(data_root, dataset=CONFIG.get("dataset"))
    
    # Verify all files are ready
    final_status = check_l1000_files(data_root, dataset=CONFIG.get("dataset"))
    if final_status['missing'] or final_status['compressed']:
        raise FileNotFoundError(
            f"Required L1000 data files are still missing or compressed after download attempt. "
            f"Missing: {len(final_status['missing'])}, Compressed: {len(final_status['compressed'])}"
        )
    logger.info("All required data files are available")

FULL_GENE_MATRIX = CONFIG["full_gene_matrix"]

2026-01-06 16:17:36 | [INFO] Checking data availability...
2026-01-06 16:17:36 | [INFO] All required files are present
2026-01-06 16:17:36 | [INFO] All files are uncompressed and ready to use
2026-01-06 16:17:36 | [INFO] All required files are present
2026-01-06 16:17:36 | [INFO] All files are uncompressed and ready to use
2026-01-06 16:17:36 | [INFO] All required data files are available


In [7]:
# Load metadata tables
inst_raw, sig_raw, cellinfo_raw, pert_raw, geneinfo, geneinfo_beta, siginfo_beta, cellinfo_beta = load_metadata_tables(PATHS)

2026-01-06 16:17:37 | [INFO]   Loading metadata tables
2026-01-06 16:17:47 | [INFO]     Loaded 473,647 signatures
2026-01-06 16:17:47 | [INFO]     Loaded 98 cell lines
2026-01-06 16:17:47 | [INFO]     Loaded 51,383 perturbations
2026-01-06 16:17:47 | [INFO]     Loaded 12,328 genes


In [8]:
NAME_TO_CVCL = {
    "HA1E": "CVCL_VU89",
    "HEK293T": "CVCL_0063",
    "HS27A": "CVCL_3719",
    "FIBRNPC": "CVCL_UK07",
    "U266": "CVCL_0566",
    "HUES3":   "CVCL_B161",
    "HUVEC":   "CVCL_2959"
}

In [9]:
sig_raw[sig_raw['cell_id'] == 'CD34'][sig_raw[sig_raw['cell_id'] == 'CD34']['pert_type'] == 'trt_cp']['pert_idose'].unique()

array(['500 nM', '1 µM', '10 µM', '3 µM', '5 µM'], dtype=object)

In [10]:
sig_raw[sig_raw['pert_type'] == 'ctl_untrt']

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
1809,BRAF001_A375_24H:CMAP-000:-666,CMAP-000,UnTrt,ctl_untrt,A375,-666.0,-666,-666,24,h,24 h,BRAF001_A375_24H_X1_B10:A09|BRAF001_A375_24H_X...
1949,BRAF001_A375_6H:CMAP-000:-666,CMAP-000,UnTrt,ctl_untrt,A375,-666.0,-666,-666,6,h,6 h,BRAF001_A375_6H_X1_B10:A09|BRAF001_A375_6H_X1_...
2087,BRAF001_HEK293T_24H:CMAP-000:-666,CMAP-000,UnTrt,ctl_untrt,HEK293T,-666.0,-666,-666,24,h,24 h,BRAF001_HEK293T_24H_X1_B10:A09|BRAF001_HEK293T...
2223,BRAF001_HEK293T_6H:CMAP-000:-666,CMAP-000,UnTrt,ctl_untrt,HEK293T,-666.0,-666,-666,6,h,6 h,BRAF001_HEK293T_6H_X1_B10:A09|BRAF001_HEK293T_...
153955,CRCGN001_HA1E_24H:CMAP-000:-666,CMAP-000,UnTrt,ctl_untrt,HA1E,-666.0,-666,-666,24,h,24 h,CRCGN001_HA1E_24H_X1_F1B4_DUO52HI53LO:A06|CRCG...
...,...,...,...,...,...,...,...,...,...,...,...,...
472375,TAK002_SW480_96H:TRCN0000000000:-666,CMAP-000,UnTrt,ctl_untrt,SW480,-666.0,-666,-666,96,h,96 h,TAK002_SW480_96H_X2_F1B6_DUO52HI53LO:A17|TAK00...
472734,TAK003_A375_96H:TRCN0000000000:-666,CMAP-000,UnTrt,ctl_untrt,A375,-666.0,-666,-666,96,h,96 h,TAK003_A375_96H_X1_B7_DUO52HI53LO:F13|TAK003_A...
472920,TAK003_HEKTE_96H:TRCN0000000000:-666,CMAP-000,UnTrt,ctl_untrt,HEKTE,-666.0,-666,-666,96,h,96 h,TAK003_HEKTE_96H_X1_B7_DUO52HI53LO:F13|TAK003_...
473106,TAK003_PC3_96H:TRCN0000000000:-666,CMAP-000,UnTrt,ctl_untrt,PC3,-666.0,-666,-666,96,h,96 h,TAK003_PC3_96H_X1_B7_DUO52HI53LO:F13|TAK003_PC...


In [11]:
sig_raw[sig_raw['sig_id'].str.startswith('BRAF001_A375_24H')]#['pert_type'].unique()

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
1683,BRAF001_A375_24H:A05,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X2_B10:A05|BRAF001_A375_24H_X...
1684,BRAF001_A375_24H:A06,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X4_B11:A06
1685,BRAF001_A375_24H:A13,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X1_B10:A13|BRAF001_A375_24H_X...
1686,BRAF001_A375_24H:A14,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X1_B10:A14
1687,BRAF001_A375_24H:A21,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X1_B10:A21
...,...,...,...,...,...,...,...,...,...,...,...,...
1817,BRAF001_A375_24H:P02,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X2_B10:P02
1818,BRAF001_A375_24H:P09,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X1_B10:P09|BRAF001_A375_24H_X...
1819,BRAF001_A375_24H:P10,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X2_B10:P10|BRAF001_A375_24H_X...
1820,BRAF001_A375_24H:P17,DMSO,DMSO,ctl_vehicle,A375,0.1,%,0.1 %,24,h,24 h,BRAF001_A375_24H_X2_B10:P17|BRAF001_A375_24H_X...


In [12]:
inst_raw[inst_raw['inst_id'].str.startswith('BRAF001_A375_24H')]#['pert_type'].unique()

,inst_id,rna_plate,rna_well,pert_id,pert_iname,pert_type,pert_dose,pert_dose_unit,pert_time,pert_time_unit,cell_id
640872,BRAF001_A375_24H_X1_B10:A13,BRAF001_A375_24H_X1,A13,DMSO,DMSO,ctl_vehicle,0.10000,%,24,h,A375
640873,BRAF001_A375_24H_X1_B10:A14,BRAF001_A375_24H_X1,A14,DMSO,DMSO,ctl_vehicle,0.10000,%,24,h,A375
640874,BRAF001_A375_24H_X1_B10:A21,BRAF001_A375_24H_X1,A21,DMSO,DMSO,ctl_vehicle,0.10000,%,24,h,A375
640875,BRAF001_A375_24H_X1_B10:B05,BRAF001_A375_24H_X1,B05,DMSO,DMSO,ctl_vehicle,0.10000,%,24,h,A375
640876,BRAF001_A375_24H_X1_B10:B06,BRAF001_A375_24H_X1,B06,DMSO,DMSO,ctl_vehicle,0.10000,%,24,h,A375
...,...,...,...,...,...,...,...,...,...,...,...
1253772,BRAF001_A375_24H_X4_B11:F06,BRAF001_A375_24H_X4,F06,BRD-K81418486,vorinostat,trt_cp,0.15625,um,24,h,A375
1253773,BRAF001_A375_24H_X4_B11:F13,BRAF001_A375_24H_X4,F13,BRD-K81418486,vorinostat,trt_cp,0.62500,um,24,h,A375
1253774,BRAF001_A375_24H_X4_B11:F14,BRAF001_A375_24H_X4,F14,BRD-K81418486,vorinostat,trt_cp,0.15625,um,24,h,A375
1253775,BRAF001_A375_24H_X4_B11:F21,BRAF001_A375_24H_X4,F21,BRD-K81418486,vorinostat,trt_cp,0.62500,um,24,h,A375


In [13]:
sig_raw[sig_raw['sig_id'].str.startswith('BRAF001_A375_24H')][sig_raw[sig_raw['sig_id'].str.startswith('BRAF001_A375_24H')]['pert_type'] == 'ctl_untrt']

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
1809,BRAF001_A375_24H:CMAP-000:-666,CMAP-000,UnTrt,ctl_untrt,A375,-666.0,-666,-666,24,h,24 h,BRAF001_A375_24H_X1_B10:A09|BRAF001_A375_24H_X...


In [14]:
sig_raw[sig_raw['cell_id'] == 'A375']['distil_id'].iloc[2]

'BRAF001_A375_24H_X1_B10:A13|BRAF001_A375_24H_X4_B11:A13'

In [15]:
sig_raw[sig_raw['cell_id'] == 'CD34'][sig_raw[sig_raw['cell_id'] == 'CD34']['pert_idose'] == '500 nM']

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
4,AML001_CD34_24H:BRD-A03772856:0.37037,BRD-A03772856,BRD-A03772856,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...
11,AML001_CD34_24H:BRD-A19500257:0.37037,BRD-A19500257,geldanamycin,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:L04|AML001_CD34_24H_X...
15,AML001_CD34_24H:BRD-A34037822:0.37037,BRD-A34037822,KUC107191N,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:N06|AML001_CD34_24H_X...
19,AML001_CD34_24H:BRD-A35207501:0.37037,BRD-A35207501,BRD-A35207501,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:H04|AML001_CD34_24H_X...
23,AML001_CD34_24H:BRD-A45664787:0.37037,BRD-A45664787,iloprost,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:J12|AML001_CD34_24H_X...
...,...,...,...,...,...,...,...,...,...,...,...,...
320,AML001_CD34_6H:BRD-K74176882:0.37037,BRD-K74176882,BRD-K74176882,trt_cp,CD34,0.37037,µM,500 nM,6,h,6 h,AML001_CD34_6H_X1_F1B10:P08|AML001_CD34_6H_X2_...
324,AML001_CD34_6H:BRD-K80368509:0.37037,BRD-K80368509,BRD-K80368509,trt_cp,CD34,0.37037,µM,500 nM,6,h,6 h,AML001_CD34_6H_X1_F1B10:N08|AML001_CD34_6H_X2_...
328,AML001_CD34_6H:BRD-K85822033:0.37037,BRD-K85822033,BRD-K85822033,trt_cp,CD34,0.37037,µM,500 nM,6,h,6 h,AML001_CD34_6H_X1_F1B10:D08|AML001_CD34_6H_X2_...
332,AML001_CD34_6H:BRD-K93754473:0.49846,BRD-K93754473,tamoxifen,trt_cp,CD34,0.49846,µM,500 nM,6,h,6 h,AML001_CD34_6H_X2_F1B10:D04|AML001_CD34_6H_X3_...


In [16]:
inst_raw[inst_raw['inst_id'].str.startswith('AML001_CD34_24H')][inst_raw[inst_raw['inst_id'].str.startswith('AML001_CD34_24H')]['pert_type'] == 'ctl_vehicle']

,inst_id,rna_plate,rna_well,pert_id,pert_iname,pert_type,pert_dose,pert_dose_unit,pert_time,pert_time_unit,cell_id
645323,AML001_CD34_24H_X1_F1B10:A05,AML001_CD34_24H_X1,A05,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645324,AML001_CD34_24H_X1_F1B10:B05,AML001_CD34_24H_X1,B05,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645325,AML001_CD34_24H_X1_F1B10:G10,AML001_CD34_24H_X1,G10,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645326,AML001_CD34_24H_X1_F1B10:H09,AML001_CD34_24H_X1,H09,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645327,AML001_CD34_24H_X1_F1B10:O02,AML001_CD34_24H_X1,O02,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645328,AML001_CD34_24H_X1_F1B10:P01,AML001_CD34_24H_X1,P01,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645329,AML001_CD34_24H_X1_F1B10:P02,AML001_CD34_24H_X1,P02,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645330,AML001_CD34_24H_X1_F1B10:P11,AML001_CD34_24H_X1,P11,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645331,AML001_CD34_24H_X3_F1B10:A06,AML001_CD34_24H_X3,A06,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34
645332,AML001_CD34_24H_X3_F1B10:B05,AML001_CD34_24H_X3,B05,DMSO,DMSO,ctl_vehicle,0.1,%,24,h,CD34


In [17]:
inst_raw[inst_raw['inst_id'].str.startswith('AML001_CD34_24H')]['pert_type'].unique()

array(['trt_cp', 'ctl_vehicle'], dtype=object)

In [18]:
inst_raw[inst_raw['cell_id'] == 'CD34']['pert_type'].unique()

array(['trt_cp', 'ctl_vehicle'], dtype=object)

In [19]:
inst_raw[inst_raw['cell_id'] == 'CD34'][inst_raw[inst_raw['cell_id'] == 'CD34']['pert_type'] == 'ctl_vehicle']

,inst_id,rna_plate,rna_well,pert_id,pert_iname,pert_type,pert_dose,pert_dose_unit,pert_time,pert_time_unit,cell_id
644982,AML001_CD34_6H_X1_F1B10:A06,AML001_CD34_6H_X1,A06,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644983,AML001_CD34_6H_X1_F1B10:B05,AML001_CD34_6H_X1,B05,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644984,AML001_CD34_6H_X1_F1B10:B06,AML001_CD34_6H_X1,B06,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644985,AML001_CD34_6H_X1_F1B10:O11,AML001_CD34_6H_X1,O11,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644986,AML001_CD34_6H_X1_F1B10:O12,AML001_CD34_6H_X1,O12,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644987,AML001_CD34_6H_X1_F1B10:P02,AML001_CD34_6H_X1,P02,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644988,AML001_CD34_6H_X1_F1B10:P11,AML001_CD34_6H_X1,P11,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644989,AML001_CD34_6H_X1_F1B10:P12,AML001_CD34_6H_X1,P12,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644990,AML001_CD34_6H_X2_F1B10:A05,AML001_CD34_6H_X2,A05,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34
644991,AML001_CD34_6H_X2_F1B10:A06,AML001_CD34_6H_X2,A06,DMSO,DMSO,ctl_vehicle,0.1,%,6,h,CD34


In [20]:
# Process cell info
cellinfo = process_cellinfo(cellinfo_raw, cellinfo_extra=cellinfo_beta)
cellinfo['cellinfo_cell_id_mixed'] = cellinfo['cellinfo_cell_id_mixed'].replace(NAME_TO_CVCL)

In [21]:
# Process perturbation metadata
pert_raw = process_pert_metadata(pert_raw, sig_raw)

In [22]:
# Annotate compounds with PubChem CIDs (optional)
if CONFIG.get("annotate_pubchem", False):
    logger.info("Mapping compounds to PubChem CIDs")
    pert_raw = annotate_pubchem_cids(pert_raw, PATHS, config=CONFIG)

2026-01-06 16:18:09 | [INFO] Mapping compounds to PubChem CIDs


2026-01-06 16:18:09 | [WARNING] Error standardizing pubchem_cid: invalid literal for int() with base 10: 'MLS003116075'


2026-01-06 16:18:09 | [INFO] Loaded cache from lincs_data/processed/pubchem_cache.json with 23168 entries
2026-01-06 16:18:09 | [INFO] Processing 20416 compounds
2026-01-06 16:18:09 | [INFO] Processed 50/20416 compounds (50 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 100/20416 compounds (100 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 150/20416 compounds (150 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 200/20416 compounds (200 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 250/20416 compounds (250 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 300/20416 compounds (300 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 350/20416 compounds (350 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 400/20416 compounds (400 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 450/20416 compounds (450 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 500/20416 compounds (500 mapped so far)
2026-01-06 16:18:09 | [INFO] Processed 550/20416 c

[16:18:09] SMILES Parse Error: syntax error while parsing: -666
[16:18:09] SMILES Parse Error: check for mistakes around position 1:
[16:18:09] -666
[16:18:09] ^
[16:18:09] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'


2026-01-06 16:18:10 | [INFO] Processed 1400/20416 compounds (1399 mapped so far)
2026-01-06 16:18:10 | [INFO] Processed 1450/20416 compounds (1449 mapped so far)
2026-01-06 16:18:10 | [INFO] Processed 1500/20416 compounds (1499 mapped so far)
2026-01-06 16:18:10 | [INFO] Processed 1550/20416 compounds (1549 mapped so far)
2026-01-06 16:18:10 | [INFO] Processed 1600/20416 compounds (1599 mapped so far)
2026-01-06 16:18:10 | [INFO] Processed 1650/20416 compounds (1649 mapped so far)
2026-01-06 16:18:10 | [INFO] Processed 1700/20416 compounds (1699 mapped so far)
2026-01-06 16:18:10 | [INFO] Processed 1750/20416 compounds (1749 mapped so far)


...

2026-01-06 16:18:18 | [INFO] Processed 20150/20416 compounds (20128 mapped so far)
2026-01-06 16:18:18 | [INFO] Processed 20200/20416 compounds (20178 mapped so far)
2026-01-06 16:18:18 | [INFO] Processed 20250/20416 compounds (20228 mapped so far)
2026-01-06 16:18:18 | [INFO] Processed 20300/20416 compounds (20278 mapped so far)
2026-01-06 16:18:18 | [INFO] Processed 20350/20416 compounds (20328 mapped so far)


[16:18:18] SMILES Parse Error: syntax error while parsing: -666
[16:18:18] SMILES Parse Error: check for mistakes around position 1:
[16:18:18] -666
[16:18:18] ^
[16:18:18] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[16:18:18] SMILES Parse Error: syntax error while parsing: -666
[16:18:18] SMILES Parse Error: check for mistakes around position 1:
[16:18:18] -666
[16:18:18] ^
[16:18:18] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[16:18:18] SMILES Parse Error: syntax error while parsing: -666
[16:18:18] SMILES Parse Error: check for mistakes around position 1:
[16:18:18] -666
[16:18:18] ^
[16:18:18] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[16:18:19] SMILES Parse Error: syntax error while parsing: restricted
[16:18:19] SMILES Parse Error: check for mistakes around position 1:
[16:18:19] restricted
[16:18:19] ^
[16:18:19] SMILES Parse Error: Failed parsing SMILES 'restricted' for input: 'restricted'
[16:18:19] SMILE

2026-01-06 16:18:24 | [INFO] Processed 20400/20416 compounds (20356 mapped so far)


[16:18:24] SMILES Parse Error: syntax error while parsing: -666
[16:18:24] SMILES Parse Error: check for mistakes around position 1:
[16:18:24] -666
[16:18:24] ^
[16:18:24] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[16:18:25] SMILES Parse Error: syntax error while parsing: -666
[16:18:25] SMILES Parse Error: check for mistakes around position 1:
[16:18:25] -666
[16:18:25] ^
[16:18:25] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[16:18:25] SMILES Parse Error: syntax error while parsing: -666
[16:18:25] SMILES Parse Error: check for mistakes around position 1:
[16:18:25] -666
[16:18:25] ^
[16:18:25] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[16:18:25] SMILES Parse Error: syntax error while parsing: -666
[16:18:25] SMILES Parse Error: check for mistakes around position 1:
[16:18:25] -666
[16:18:25] ^
[16:18:25] SMILES Parse Error: Failed parsing SMILES '-666' for input: '-666'
[16:18:26] SMILES Parse Error: syntax er

2026-01-06 16:18:27 | [INFO] Mapped 20364 out of 20416 compounds to PubChem CIDs


In [23]:

# Process signature metadata
sig = process_sig_metadata(sig_raw, cellinfo, pert_raw, CONFIG, PATHS)

2026-01-06 16:18:27 | [INFO]   Processing signature metadata


In [38]:
logger.info('  Building .obs dataframe')
obs = build_obs_dataframe(sig, dataset=CONFIG.get("dataset"))

2026-01-06 16:21:38 | [INFO]   Building .obs dataframe


In [39]:
obs

,lincs_sig_id,distil_id,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_dose,...,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,source_gctx,sample_id,pert_type_init
sig_id,,,,,,,,,,,,,,,,,,,,,
AML001_CD34_24H:A05,0,AML001_CD34_24H_X1_F1B10:A05,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:A06,1,AML001_CD34_24H_X3_F1B10:A06,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:B05,2,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:B06,3,AML001_CD34_24H_X3_F1B10:B06,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,unknown,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle
AML001_CD34_24H:BRD-A03772856:0.37037,4,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...,None,None,CD34,BRD-A03772856,compound,False,0.37037,0.37037 µM,...,unknown,human,NaN,None,3237298,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_BRD-A03772856_CD34,trt_cp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TAK004_U2OS_96H:TRCN0000370007:1,473642,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:J10|TAK004_U...,None,None,CVCL_0042,WWTR1,genetic,False,NaN,1.0 µL,...,15-year-old stage,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_WWTR1_CVCL_0042,trt_sh
TAK004_U2OS_96H:TRCN0000370678:1,473643,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:E02|TAK004_U...,None,None,CVCL_0042,GRB10,genetic,False,NaN,1.0 µL,...,15-year-old stage,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_GRB10_CVCL_0042,trt_sh
TAK004_U2OS_96H:TRCN0000370697:1,473644,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:A18|TAK004_U...,None,None,CVCL_0042,GRB10,genetic,False,NaN,1.0 µL,...,15-year-old stage,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_GRB10_CVCL_0042,trt_sh


In [40]:
obs['sig_id'] = obs.index

In [41]:
logger.info('  Enforcing strict obs schema')
obs_for_schema = enforce_obs_schema(obs)

2026-01-06 16:22:19 | [INFO]   Enforcing strict obs schema


In [42]:
logger.info('  Processing gene annotations')
var = process_gene_annotations(geneinfo, geneinfo_beta, FULL_GENE_MATRIX)

2026-01-06 16:22:21 | [INFO]   Processing gene annotations
2026-01-06 16:22:21 | [INFO]   Restricting to 978 landmark genes


In [43]:
obs

,lincs_sig_id,distil_id,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_dose,...,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,source_gctx,sample_id,pert_type_init,sig_id
sig_id,,,,,,,,,,,,,,,,,,,,,
AML001_CD34_24H:A05,0,AML001_CD34_24H_X1_F1B10:A05,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle,AML001_CD34_24H:A05
AML001_CD34_24H:A06,1,AML001_CD34_24H_X3_F1B10:A06,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle,AML001_CD34_24H:A06
AML001_CD34_24H:B05,2,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle,AML001_CD34_24H:B05
AML001_CD34_24H:B06,3,AML001_CD34_24H_X3_F1B10:B06,None,None,CD34,DMSO,compound,True,0.00000,0.1 %,...,human,NaN,None,679,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_DMSO_CD34,ctl_vehicle,AML001_CD34_24H:B06
AML001_CD34_24H:BRD-A03772856:0.37037,4,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...,None,None,CD34,BRD-A03772856,compound,False,0.37037,0.37037 µM,...,human,NaN,None,3237298,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_BRD-A03772856_CD34,trt_cp,AML001_CD34_24H:BRD-A03772856:0.37037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TAK004_U2OS_96H:TRCN0000370007:1,473642,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:J10|TAK004_U...,None,None,CVCL_0042,WWTR1,genetic,False,NaN,1.0 µL,...,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_WWTR1_CVCL_0042,trt_sh,TAK004_U2OS_96H:TRCN0000370007:1
TAK004_U2OS_96H:TRCN0000370678:1,473643,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:E02|TAK004_U...,None,None,CVCL_0042,GRB10,genetic,False,NaN,1.0 µL,...,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_GRB10_CVCL_0042,trt_sh,TAK004_U2OS_96H:TRCN0000370678:1
TAK004_U2OS_96H:TRCN0000370697:1,473644,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:A18|TAK004_U...,None,None,CVCL_0042,GRB10,genetic,False,NaN,1.0 µL,...,human,female,Caucasian,None,None,None,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,None_None_GRB10_CVCL_0042,trt_sh,TAK004_U2OS_96H:TRCN0000370697:1


In [44]:
logger.info('  Matching obs to GCTX column IDs')
obs_helpers = build_obs_helpers(obs, PATHS["level5_gctx"])

2026-01-06 16:22:25 | [INFO]   Matching obs to GCTX column IDs
2026-01-06 16:22:25 | [INFO]   Loaded 473,647 column IDs from GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx
2026-01-06 16:22:25 | [INFO]   Using obs index to subset GCTX (473647/473647 samples match)


In [45]:
obs_helpers

,source_gctx,gctx_id
sig_id,,
AML001_CD34_24H:A05,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:A05
AML001_CD34_24H:A06,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:A06
AML001_CD34_24H:B05,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:B05
AML001_CD34_24H:B06,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:B06
AML001_CD34_24H:BRD-A03772856:0.37037,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,AML001_CD34_24H:BRD-A03772856:0.37037
...,...,...
TAK004_U2OS_96H:TRCN0000370007:1,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,TAK004_U2OS_96H:TRCN0000370007:1
TAK004_U2OS_96H:TRCN0000370678:1,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,TAK004_U2OS_96H:TRCN0000370678:1
TAK004_U2OS_96H:TRCN0000370697:1,lincs_data/GSE92742_Broad_LINCS_Level5_COMPZ.M...,TAK004_U2OS_96H:TRCN0000370697:1


In [46]:
sig_raw[sig_raw['pert_type'].isin(['trt_cp']) | (sig_raw['pert_type'].isin(['ctl_vehicle']) & sig_raw['pert_iname'].isin(['DMSO']))]

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
0,AML001_CD34_24H:A05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:A05
1,AML001_CD34_24H:A06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:A06
2,AML001_CD34_24H:B05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...
3,AML001_CD34_24H:B06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:B06
4,AML001_CD34_24H:BRD-A03772856:0.37037,BRD-A03772856,BRD-A03772856,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...
...,...,...,...,...,...,...,...,...,...,...,...,...
469552,RAD001_PC3_6H:P02,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P02
469553,RAD001_PC3_6H:P13,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P13
469554,RAD001_PC3_6H:P23,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P23
469555,RAD001_PC3_6H:P24,DMSO,DMSO,ctl_vehicle,PC3,-666.0,-666,-666,6,h,6 h,RAD001_PC3_6H_X2_F1B5_DUO52HI53LO:P24


In [47]:
logger.info('  Extracting expression from GCTX')
expr = load_expression(obs_helpers, PATHS["level5_gctx"], gene_ids=var.index.astype(str))

2026-01-06 16:22:33 | [INFO]   Extracting expression from GCTX


/home/ubuntu/venv/lib/python3.12/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
/home/ubuntu/venv/lib/python3.12/site-packages/cmapPy/pandasGEXpress/parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


In [48]:
logger.info('  Assembling AnnData')
padata_processed = assemble_anndata(obs_for_schema, var, expr)

2026-01-06 16:22:40 | [INFO]   Assembling AnnData


/home/ubuntu/venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1774: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [49]:
logger.info(f'  Assembled AnnData: {padata_processed.n_obs:,} × {padata_processed.n_vars:,}')
logger.info('L1000-specific processing completed')

2026-01-06 16:22:58 | [INFO]   Assembled AnnData: 473,647 × 978
2026-01-06 16:22:58 | [INFO] L1000-specific processing completed


In [50]:
padata_processed.obs

,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,...,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,distil_id,sig_id,pert_type_init,pert_dose
sample_id,,,,,,,,,,,,,,,,,,,,,
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X1_F1B10:A05,AML001_CD34_24H:A05,ctl_vehicle,0.1 %
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X3_F1B10:A06,AML001_CD34_24H:A06,ctl_vehicle,0.1 %
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...,AML001_CD34_24H:B05,ctl_vehicle,0.1 %
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.00000,24.0,cell,bone,...,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X3_F1B10:B06,AML001_CD34_24H:B06,ctl_vehicle,0.1 %
None_None_BRD-A03772856_CD34,NaN,NaN,CD34,BRD-A03772856,compound,False,0.37037,24.0,cell,bone,...,human,unknown,unknown,3237298,-666,-666,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...,AML001_CD34_24H:BRD-A03772856:0.37037,trt_cp,0.37037 µM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
None_None_WWTR1_CVCL_0042,NaN,NaN,CVCL_0042,WWTR1,genetic,False,NaN,96.0,cell,bone,...,human,female,Caucasian,NaN,-666,-666,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:J10|TAK004_U...,TAK004_U2OS_96H:TRCN0000370007:1,trt_sh,1.0 µL
None_None_GRB10_CVCL_0042,NaN,NaN,CVCL_0042,GRB10,genetic,False,NaN,96.0,cell,bone,...,human,female,Caucasian,NaN,-666,-666,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:E02|TAK004_U...,TAK004_U2OS_96H:TRCN0000370678:1,trt_sh,1.0 µL
None_None_GRB10_CVCL_0042,NaN,NaN,CVCL_0042,GRB10,genetic,False,NaN,96.0,cell,bone,...,human,female,Caucasian,NaN,-666,-666,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:A18|TAK004_U...,TAK004_U2OS_96H:TRCN0000370697:1,trt_sh,1.0 µL


In [51]:
# Save the assembled dataset
logger.info(f"Saving assembled dataset to: {config['output_file']}")
padata_processed.write_h5ad(config['output_file'], compression='gzip')
logger.info(f"{config['output_file']} assembly complete")

2026-01-06 16:22:58 | [INFO] Saving assembled dataset to: level5_phase1_not_filtered.h5ad
2026-01-06 16:24:10 | [INFO] level5_phase1_not_filtered.h5ad assembly complete


In [34]:
padata_processed[padata_processed.obs['cell_type'] == 'CVCL_0566']

View of AnnData object with n_obs × n_vars = 273 × 978
    obs: 'plate', 'well', 'cell_type', 'perturbagen', 'pert_type', 'is_control', 'pert_dose_uM', 'pert_time_h', 'suspension_type', 'tissue', 'tissue_type', 'disease', 'library', 'stimulation', 'guide', 'dataset', 'assay', 'development_stage', 'organism', 'sex', 'self_reported_ethnicity', 'pubchem_cid', 'psbulk_cells', 'psbulk_counts', 'distil_id', 'sig_id'
    var: 'symbol'

In [42]:
padata_processed.obs[padata_processed.obs['is_control'] == True]

,plate,well,cell_type,perturbagen,pert_type,is_control,pert_dose_uM,pert_time_h,suspension_type,tissue,...,development_stage,organism,sex,self_reported_ethnicity,pubchem_cid,psbulk_cells,psbulk_counts,distil_id,sig_id,pert_type_init
sample_id,,,,,,,,,,,,,,,,,,,,,
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,unknown,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X1_F1B10:A05,AML001_CD34_24H:A05,ctl_vehicle
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,unknown,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X3_F1B10:A06,AML001_CD34_24H:A06,ctl_vehicle
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,unknown,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...,AML001_CD34_24H:B05,ctl_vehicle
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,unknown,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X3_F1B10:B06,AML001_CD34_24H:B06,ctl_vehicle
None_None_DMSO_CD34,NaN,NaN,CD34,DMSO,compound,True,0.0,24.0,cell,bone,...,unknown,human,unknown,unknown,679,-666,-666,AML001_CD34_24H_X1_F1B10:A05|AML001_CD34_24H_X...,AML001_CD34_24H:DMSO:0.1,ctl_vehicle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
None_None_lacZ_CVCL_0042,NaN,NaN,CVCL_0042,lacZ,genetic,True,0.0,96.0,cell,bone,...,15-year-old stage,human,female,Caucasian,NaN,-666,-666,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:D10|TAK004_U...,TAK004_U2OS_96H:TRCN0000072237:1,ctl_vector
None_None_LUCIFERASE_CVCL_0042,NaN,NaN,CVCL_0042,LUCIFERASE,genetic,True,0.0,96.0,cell,bone,...,15-year-old stage,human,female,Caucasian,NaN,-666,-666,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:M19|TAK004_U...,TAK004_U2OS_96H:TRCN0000072246:1,ctl_vector
None_None_LUCIFERASE_CVCL_0042,NaN,NaN,CVCL_0042,LUCIFERASE,genetic,True,0.0,96.0,cell,bone,...,15-year-old stage,human,female,Caucasian,NaN,-666,-666,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:M14|TAK004_U...,TAK004_U2OS_96H:TRCN0000072248:1,ctl_vector


In [46]:
sig_raw[sig_raw['pert_iname'] == 'LUCIFERASE']

,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
38964,CNS001_A375_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,A375,-666.0,-666,-666,96,h,96 h,DER001_A375_96H:TRCN0000072248:-666|KDA001_A37...
38975,CNS001_A549_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,A549,-666.0,-666,-666,96,h,96 h,KDB004_A549_96H:TRCN0000072254:-666|KDA001_A54...
38983,CNS001_ASC_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,ASC,-666.0,-666,-666,96,h,96 h,KDB001_ASC_96H:TRCN0000072250:-666|KDB008_ASC_...
39004,CNS001_HA1E_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,HA1E,-666.0,-666,-666,96,h,96 h,KDA010_HA1E_96H:TRCN0000072266:-666|KDA001_HA1...
39015,CNS001_HCC515_96H:LUCIFERASE:-666,CNS001-LUCIFERASE,LUCIFERASE,ctl_vector.cns,HCC515,-666.0,-666,-666,96,h,96 h,DER001_HCC515_96H:TRCN0000072248:-666|KDA005_H...
...,...,...,...,...,...,...,...,...,...,...,...,...
473006,TAK003_HEKTE_96H:TRCN0000072266:-666,TRCN0000072266,LUCIFERASE,ctl_vector,HEKTE,-666.0,-666,-666,96,h,96 h,TAK003_HEKTE_96H_X1_B7_DUO52HI53LO:H21|TAK003_...
473192,TAK003_PC3_96H:TRCN0000072266:-666,TRCN0000072266,LUCIFERASE,ctl_vector,PC3,-666.0,-666,-666,96,h,96 h,TAK003_PC3_96H_X1_B7_DUO52HI53LO:H21|TAK003_PC...
473450,TAK004_U2OS_96H:TRCN0000072246:1,TRCN0000072246,LUCIFERASE,ctl_vector,U2OS,1.0,µL,1 µL,96,h,96 h,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:M19|TAK004_U...
473451,TAK004_U2OS_96H:TRCN0000072248:1,TRCN0000072248,LUCIFERASE,ctl_vector,U2OS,1.0,µL,1 µL,96,h,96 h,TAK004_U2OS_96H_X1_B6_DUO52HI53LO:M14|TAK004_U...
